**KI-Nutzung und Herkunft**

KI-Unterstützung: ChatGPT und teilweise Claude gemäß Erklärung der Arbeit; Inhalte durch den Verfasser bearbeitet. Diese Kopie ergänzt ausschließlich Kennzeichnungen durch Codex; historische Ausgaben und Berechnungsanweisungen bleiben erhalten.

Originaldatei: `FINAL_URL_SSL_INTEGRATED_RESEARCH_ALL_IN_ONE_v2_COMPLETE_COVERAGE(1).ipynb`.



# Finale integrierte Untersuchung: Self-Supervised Representation Learning auf URL- und HTML-Daten

Dieses Notebook führt die bislang getrennten experimentellen Stränge in **einem durchgängigen Forschungsablauf**
zusammen. Ausgangspunkt ist die nachträglich identifizierte modalitätsspezifische Erweiterung des
Self-Supervised Representation Learning (SSRL): Neben dem bereits etablierten HTML-Text-DAPT wird nun auch
der URL-Encoder auf denselben 200.000 ungelabelten Webseiten mittels Masked Language Modeling (MLM)
domänenspezifisch angepasst.

## Forschungslogik

### FF1 — Downstream-Nutzen selbstüberwachter Repräsentationen
Es wird ein faktorielles 2×2-Design verwendet:

| Repräsentation | URL | HTML-Text |
|---|---|---|
| **R0** | vortrainiertes BERT | vortrainiertes RoBERTa |
| **RU** | **URL-DAPT-BERT** | vortrainiertes RoBERTa |
| **RT** | vortrainiertes BERT | **HTML-DAPT-RoBERTa** |
| **RUT** | **URL-DAPT-BERT** | **HTML-DAPT-RoBERTa** |

Dadurch können der isolierte URL-DAPT-Effekt, der isolierte HTML-DAPT-Effekt, deren kombinierter Nutzen
und die Interaktion zwischen beiden Modalitätsanpassungen bestimmt werden. Die bereits ausgeführten
contrastiven Varianten bleiben als unverändertes Negativergebnis eines anderen SSL-Lernziels erhalten.

### FF2 — Labelverfügbarkeit und Downstream-Mechanismus
R0, RU, RT und RUT werden über feste Labelbudgets mit linearem Probe und kleinem MLP untersucht.
Zusätzlich wird RUT im bestehenden end-to-end Deep-DUAL-System bei 20k und 200k Labels geprüft und mit
den historischen, reproduzierten R0-/RT-Bedingungen gepaart.

### FF3 — Generalisierung und niedrige False-Positive-Raten
Die historische N10-Replikation von R0 und RT bleibt unverändert. Mit **denselben zehn Model-Seeds und
denselben zehn Label-Rank-Seeds** werden RU und RUT ergänzt. Damit können auf OFFICIAL, Domain-OOD,
Template-OOD, Domain+Template-OOD und Late-Q4 neue gepaarte Kontraste berechnet werden. Der zentrale
Erweiterungskontrast ist **RUT − RT**: der zusätzliche URL-DAPT-Nutzen bei bereits adaptiertem HTML.

### FF4 — Engineering
Die Repräsentationswahl erfolgt ausschließlich auf DEV. Der gewählte Champion wird danach unverändert
für DUAL/TRI, DOM-Gate, Gated Fusion und URL-first-Kaskade verwendet. Erst nach dem DEV-only
Architecture Freeze werden CAL und FINAL technisch freigeschaltet.

## Transparenz

Die URL-SSL-Erweiterung wurde nach Kenntnis früherer FINAL-Ergebnisse entwickelt. Sie ist daher eine
**post-hoc modalitätsspezifische Erweiterung** und wird nicht rückwirkend als ursprünglich präregistrierte
Untersuchung dargestellt. Innerhalb dieses neuen Laufs werden jedoch alle nachfolgend festgeschriebenen
Bedingungen vollständig ausgeführt; optionales Stoppen anhand neuer Ergebnisse ist ausgeschlossen.


## Vollständigkeitsziel v2

Die v2 repliziert zusätzlich die **Ergebnisbreite der bisherigen v7/v8-Forschung**:
alle sieben Labelbudgets, Marginal Label Utility, B95, historische XGBoost-Systemreferenz,
Deep-N3, alle niedrigen FPR-Betriebspunkte, alle fünf Shift-Szenarien, DUAL/TRI-Branchvergleiche,
mittlere Fusion-Gewichte, DOM-Gate, die historisch definierte einzelne URL-first-Kaskade,
vollständige Operational-Metriken einschließlich VRAM/GPU-Telemetrie und die 1-Million-Seiten-Illustration.

Es werden **keine künstlichen zusätzlichen Kaskadenvarianten** eingeführt, die in der bisherigen
Forschung nicht Bestandteil des eingefrorenen Designs waren.


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 00 — Fester Integrationsplan, Konfiguration und atomare Hilfsfunktionen
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, re, json, math, time, random, hashlib, itertools, shutil, warnings
import zipfile, gzip, base64, io, threading, subprocess
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, IterableDataset, DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    roc_curve, roc_auc_score
)
from scipy.stats import t as student_t, beta as beta_dist

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

if not torch.cuda.is_available():
    raise RuntimeError("GPU erforderlich; Zielumgebung: Kaggle Tesla T4.")
DEVICE = torch.device("cuda")
AMP = True

VERSION = "FINAL_URL_SSL_INTEGRATED_RESEARCH_ALL_IN_ONE_v2_COMPLETE_COVERAGE"
MASTER_SEED = 20260817

# -----------------------------
# Self-Supervised Pretraining
# -----------------------------
URL_MODEL_ID = "bert-base-uncased"
TEXT_MODEL_ID = "roberta-base"
URL_MAX_LEN = 128
TEXT_MAX_LEN = 256

URL_DAPT_ROWS = 200_000
URL_DAPT_EPOCHS = 1
URL_DAPT_BATCH_CANDIDATES = [16, 8]
URL_DAPT_LR = 5e-5
URL_DAPT_WEIGHT_DECAY = .01
URL_DAPT_WARMUP_FRAC = .06
URL_MLM_PROB = .15
BERT_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"

# -----------------------------
# FF1 modal representation screen
# Reproduces the new 200k/N5 DEV-only modality design.
# -----------------------------
FF1_MODEL_SEEDS = [202, 212, 222, 232, 242]
FF1_LABEL_RANK_SEEDS = [9201, 9202, 9203, 9204, 9205]
FF1_BUDGET = 20_000

# -----------------------------
# FF2 label efficiency
# Existing v8 budget grid, now extended to all 4 DAPT modalities.
# -----------------------------
BUDGETS = [2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000]
CURVE_MODEL_SEEDS = [42, 62, 82, 102, 122]
CURVE_LABEL_RANK_SEEDS = [20260813, 20260823, 20260833, 20260843, 20260853]
RETENTION_TARGET = .95

# Deep-N3 extension: historical R0/RT are embedded; new RUT uses same seeds/sample.
DEEP_N3_SEEDS = [42, 82, 122]
DEEP_N3_BUDGETS = [20_000, 200_000]
DEEP_N3_RANK_SEED = 20260813

# -----------------------------
# FF3 N10
# Same pairings as historical N10, now adding RU and RUT.
# -----------------------------
N10_MODEL_SEEDS = [142, 162, 182, 202, 222, 242, 262, 282, 302, 322]
N10_LABEL_RANK_SEEDS = [9101, 9102, 9103, 9104, 9105, 9106, 9107, 9108, 9109, 9110]
N10_BUDGET = 20_000

SCENARIOS = [
    "OFFICIAL_TEST",
    "DOMAIN_OOD_EXACT",
    "TEMPLATE_OOD_EXACT",
    "DOMAIN_TEMPLATE_OOD_EXACT",
    "LATE_TEST_Q4",
]
OOD_FAMILY = SCENARIOS[1:]

# -----------------------------
# Low-FPR evaluation
# -----------------------------
TARGET_FPRS = [.0001, .0005, .001, .0025, .005, .01, .02]
PRIMARY_FPR = .005

# -----------------------------
# Deep architecture: frozen historical hyperparameters
# -----------------------------
PROJ_DIM = 256
DEEP_LAST_N = 4
DEEP_EPOCHS = 1
ENCODER_LR = 1e-5
HEAD_LR = 2e-4
WEIGHT_DECAY = .01
WARMUP_RATIO = .05
AUX_TOTAL_WEIGHT = .30
DUAL_BATCH_CANDIDATES = [16, 8, 4]
TRI_BATCH_CANDIDATES = [8, 4, 2]
SCORE_BATCH_CANDIDATES = [128, 96, 64, 48, 32, 16]

# Engineering follows the historical seed and anchor.
ENGINEERING_SEED = 82
ENGINEERING_RANK_SEED = 20260813

# DOM
DOM_MAX_NODES = 512
DOM_DIM = 128
DOM_LAYERS = 3
DOM_COLS = ["dom_tag", "dom_parent_idx", "dom_depth", "dom_attr_count", "dom_child_count"]

# Cascade gate (same historical rule, re-calibrated on new champion).
CASCADE_MAX_TPR_LOSS_PP = 1.0
CASCADE_MIN_FULL_REDUCTION = .30

# Probe configuration
EMBED_BATCH_URL = 128
EMBED_BATCH_TEXT = 64
MLP_EPOCHS = 12
MLP_BATCH = 256

# Operational benchmark
BENCH_B1_N = 500
BENCH_TPUT_N = 5000
BENCH_REPEATS = 3
BENCH_BATCH = 32
GPU_POLL_S = .10

# Final-system uncertainty
BOOTSTRAP_REPS = 1000
BOOTSTRAP_SEED = 20260817

# Multi-session resume is transparent and condition-level.
AUTO_RESUME_FROM_KAGGLE_INPUT = True
KEEP_N10_MODEL_STATES_AFTER_SCORING = False

ROOT = Path("/kaggle/working/final_url_ssl_integrated_v2") if Path("/kaggle/working").exists() else Path("/mnt/data/final_url_ssl_integrated_v2")
CKPT = ROOT/"checkpoints"
EMB = ROOT/"embeddings"
SCORES = ROOT/"scores"
RESULTS = ROOT/"results"
AUDIT = ROOT/"audit"
META = ROOT/"meta"
TABLES = ROOT/"thesis_tables"
for p in [ROOT, CKPT, EMB, SCORES, RESULTS, AUDIT, META, TABLES]:
    p.mkdir(parents=True, exist_ok=True)

def seed_all(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))

def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path)+".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    os.replace(tmp, path)

def atomic_torch(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path)+".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def amp_ctx():
    return torch.autocast("cuda", dtype=torch.float16) if AMP else nullcontext()

seed_all(MASTER_SEED)

PROTOCOL = {
    "version": VERSION,
    "scientific_status": "POST_HOC_MODALITY_SPECIFIC_INTEGRATION_AFTER_HISTORICAL_FINAL_KNOWN",
    "historical_final_known": True,
    "no_optional_stopping": True,
    "all_planned_blocks_must_be_reported": True,
    "representations": {
        "R0": "URL_BASE + TEXT_BASE",
        "RU": "URL_DAPT + TEXT_BASE",
        "RT": "URL_BASE + TEXT_DAPT",
        "RUT": "URL_DAPT + TEXT_DAPT",
    },
    "ff1": {
        "design": "2x2 URL-DAPT x HTML-DAPT, N5, DEV-only",
        "budget": FF1_BUDGET,
        "model_seeds": FF1_MODEL_SEEDS,
        "label_rank_seeds": FF1_LABEL_RANK_SEEDS,
        "contrastive_evidence": "historical v7.2 evidence retained unchanged",
    },
    "ff2": {
        "budgets": BUDGETS,
        "reps": ["R0","RU","RT","RUT"],
        "probes": ["LINEAR","MLP"],
        "deep_n3_new_rep": "RUT",
        "deep_n3_historical_refs": ["R0","RT"],
    },
    "ff3": {
        "n10_new_reps": ["RU","RUT"],
        "n10_historical_refs": ["R0","RT"],
        "model_seeds": N10_MODEL_SEEDS,
        "label_rank_seeds": N10_LABEL_RANK_SEEDS,
        "primary_extension_contrast": "RUT-RT on OFFICIAL_TEST",
        "secondary_extension_family": "RUT-RT across exactly four OOD/Late scenarios with Holm correction",
    },
    "ff4": {
        "representation_selection": "DEV-only N5 linear-probe mean TPR at 0.5% FPR; P@R90/AP tie-breaks",
        "dom_gate": "+0.5pp TPR@0.5% FPR OR TRI FPR@TPR90 <= 90% of DUAL",
        "cascade_gate": {
            "max_tpr_loss_pp": CASCADE_MAX_TPR_LOSS_PP,
            "min_full_reduction": CASCADE_MIN_FULL_REDUCTION,
            "max_fpr_increase": .001,
        },
    },
    "access_order": "SSL/DEV -> ARCHITECTURE_FREEZE -> CAL/FINAL",
}
atomic_json(AUDIT/"INTEGRATED_PROTOCOL.json", PROTOCOL)
print(torch.cuda.get_device_name(0))
print(json.dumps(PROTOCOL, indent=2, ensure_ascii=False))


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 00b — Unveränderliche historische Evidenz einbetten und per SHA-256 validieren
# Diese Dateien werden nicht neu interpretiert oder verändert; sie dienen der exakten
# Kontinuität zu den bereits reproduzierten v7/v8/N10-Ergebnissen.
EMBEDDED_ASSETS_B64_GZ = {'HIST_N10_ALL.csv': 'H4sIANjCgmoC/5WcSY8cR5KF7/wtgYTvy5HolgAB6pamR4e5EWyxeloYjURQnP8/3zP3KKZTEVFZpEiqstIr3NyW92zx/PT08ddffn7/+Wn7398/PP367o+npw/br+//yf9+ev/b/4yvPz193P75fx/+++nz9uH95/d/8O8fPz/99v7TL79vn99/4vV3//r4afv8709Pf/z7918/bJ/5Sq98/PT08y9//PL7b7zCC9vn37Z/8d/Hdx+fPr3zzrnt7Y/bj+/ef373j+7euM2nsHXv/PYPtwWn73/73d/ffr/98O233/3lu7ffv/vpm//8aXM35/IWbj3EFpoP2dmvxDdKSz7HknKL+pXGe0strrvQfaqp81Lni1hLZgc5pVK2HHKKW/Zt666yh5BCrlu+LQtd1dIWasqhRR+SDyHqpVJDqJEf6Iq2cSHHX3/429vv/v7uhx/++u6b/3r7l2tReipZj44p8Dy91bvcfSipxNyQ0STxhUdndhljKA0ZeihbivxxNfL00ELevLstK9MQpXYfeQzfCCE1vVSLdwhSew0953Ahyk/f/O3H79/+9M2DwjS04Eqq2bXYp1ZCjs0VNJCaPbz1jPpaLamVzGmnxo43VmytsyNk4c3Syv1CH2ypzy0gkEvBd29Kyc23xo5r5seGl5XySoG667lWn3usLdr+UU9CQ7n6JjnrUI8vhZextOo7SqkSJElFLvW0Bc9yqWdZGeNQD5vNsfdea2aF1JOxU44n828P7UIok0Te8u4/0pUcPLJnbDf6xPNatvdiMvhPrZxvbMPM2EtoHDcm2QL7xtHyxtZ7LJX9S5Z2W5ZF02nPOEmo3fEvfjOcz+FPoebk2MgqhH/A7XNuOvUpxJAiheRay4nT5ledBiZ7qL3jvC6OJ2NHssMgv8KUqvNVLmNu7zff0aUM7H6hGwaWPX6TU0NUFG/KiAEl4HU141X5Qo5Ttz8SJXvCSWypc9y+hGFYPfqYvfc8vI5zDdhLRBu1FN665YLjbyk7+b0vG6rMXYa1rPS7Ahzfrw2VlqGTKg2zOhALSukXsly4yZE0kYhTsYfQFQB2zyeoucpvvjEVg8AFY7e/UUwIwW/yIXk+DtJKt3h8v7AMxURpFbvWy2EEMQJzrZhrR5ExvKyYV8qUCujB4ZWaiJVlaKgigg+RcOzzjMwKtaV7FOlwfQwImRJugeu3sHk5szS0rExDKBajix6jK0PflRMgphHSm4+9XMh06PlHYnQwrqDvhjAD3JzrYA1RxhNEvSvD8zEWhWPwUnEOXJCv4zIxlw1nz1u/LavS8PsqXQFPxJahFpTcMemkgEm0eYO/FRMhPAL38YaneUJUXWAFz9dLO6L4iEezE7x3PrLKPTgxcCdgV5XNZx/N4UGU5JBSiHK3MNhS4hsWlfgaZO3DSTBkwgSwA3Vo6UKAE4c/lqE3Ha6vQAqRfwI9/w/YEwhK9H6YUyqoCfeoXTaRkRuHh8/I4QWOuQyHv1+Z/TAnCRNaVwQMw8KA1k6gd6XgSeVCllPnOJamIgu2KioE7LXxXs61JOme4x/Y1jEcNgDS40kJqGcjmwJZ03GIthCY8m1Z6AZLiMl5fCDl6NIwtlpQMHtBUdjzy3p5pUjQCHTfRRVj9HUoqNz/nuCCBwvfPZoQ1LMh/D2ZvxegHvuUgpaVk1S2UhSkC+EBNx+MRvyMqO1gaPFCpgN/PxSjQmoRBEuXtw5gEUXMwHIJLsXht+gFNplj0vqNqApOFnl7BVSgwqD+7X4RLj7CXcG95Tzwxj78pRMviJLIqMixiPAIzgNjVRH8Xoam2B77pPdxWhfEEFKCjRPjhq/AZ4oTVSHulQ0GlgUnSW7P1wQl/Abrul9YhtsT3UFMBAP6ByduuSF9B4FYXS7EOIX5I0mIq3BsGJVoih2h6D3YCzMiwjzDPNTPiV/gBGjAuCxGlY3esxVQxbx+WdmGKuG/MZDkgMAzKjciCPIiG9bl04UsF5B4qBfQFtPnoDH6OvkXNPcL1o8dsR1IUwtwDeJwVsZFFJDTcwQgP8/Pt2WdM+uCp+A+AG+DuQxRIHsAJLQiRUjYy2p5pURgbxWcdcymGZo4UeAG382EnTgsHKJWYycyIQG0GH4o9QjnxROF0Vgk6rlf6Mo0Upyx8H0w3A/bI+ZhdJDOEgkNFyIdYvyBFMRK8TqADZPBzIbP8yr+Ax3HNIYYBNCAZ9SKqRB3AQ50WJO83m9Fu6i3ZdXASAU85FdAIRCMHwU7kIRwfhK3NwS8ZiLExzAen60436IJUCzYa9PbFSqxayV3E+ThL3yNR2Ut0SlnSxvl7STzQYws35aFExgJWQ1uo8wGkzMswfo4DBgKIOrLhQSnIH8kBIDG46FagfgS0jAnjtS3CA7mZ5CH3ZUMBDYnl8Uh5OrJsvmgxxu3N8V+WVlm9kyuQmSL0H3Xd1ZvAT8qP7lSxgUgHgmDDyJGCiAiLuKnt3NeQD+QVYazE2O8vIKXzSvAUP0dzdtRCzmJuNf9uqkDwnbH/xvLeNJQi1I2yA9SNv+IWl4pEvwPszZQI4fcqy1YMawrRGHw0A9RgXiMn5HXJEkTFI7N3+HEyJvN35eVbsZeVAgb9ioX1CEUSZg8RowJGngh1CHIH8kRPUCFYpQFu2Fn2BKElXjJEdfBoXScRcCmKkoWzENsNjZlMKjCHETstiybmIK9wZKlWD+KNsLaqjpML+lrxTyC8trcc8zaE6yMWe01vL2Il7Ai0A5m0+tEE2Jl5DUyROeVzeNj+L2T3xcSJJxc5aJlYRsGVhxahHui8DbdB1qB1wcVXMhaL+Q4hfkjUVTTISwRFUtyrZ5V8ULMgDS7VJRg05Amd1fFIywQsI6reApy4Dn4yI+Y5J5IHkZiR24cLmS5AMUjaSJhCZbB5shQ/Q7zED++xixin4leJaEKicgNDVM2H5X1NiP3rPIyPCnmfuF0EjLjnovgyo0qC95uR4i7EeXDy3p5pUjJKqyk8Q3imicPi2AekRh+FiZCYu0ceGHr8nt4i/w+GLeHr+MDw++XhWXYGjCJUqWbVkcmrOIOrLhWxe4rkQ5x/kgKOUBPCdKrNGiUWcgeoE9AOJGtTm5PeJBTJwUrAiqmBazqLZuhab3dr5ka6KLHTXU8FDWJXBTcpvl3eRPZtUmQHoP5EBcZRuKIJZU9lU9KA6Fg4ED0M85E1eP45Ult0YCSRdLzTYDA87MKdem2rAsT5HFtQi8Bqs7kCs6GLaNTr6psu9j/KcgfiUC2o0ooB+jlIaOiEsgcXeA/AH1yeszbYzhVbMYK3ABhFESKiWwKRCqqLAv9JIjiJFAtoGNKAufG+WvIRLmWLiS5AMQjWcTjCEqi9Mg0XJ1wCxjAdyvuPYOPyG9WEY5dEK5KLRCWYK5eVWDxOEa6LQtHbQbEgB6rAhxTGd6Pkhx+E3gV/Cwvq+WVMqnCAtcu7EqV4gGOrSpxr4lkIg+MgzACKUHMpSlBKWq/REhBUo1etQnoVbstC2dSApLzh2iRcIu9cEemQtaJ9DleiHQI8UdSEHnFD+W+LU10VMdLj8SJy57Ig8dd5XqoCkEY3guoF4+3s1c4NYBfbsuyOvISUjTCnJKTulfsg2IH1gdVTqta/CPervr1VwHL2SPXgn0ScOX979ndwnmjZZZNSBK7civCNF6vYBiUmqXbsnBCPAFLZLpGVcoHLMrl8BM0C9FMF3Kce/2BKJ5swbE/TpoUfVS8IHWcILSpq9Y5zlDUPGWQBmgLiIKZb7FbJk+WRZIC0+23ZeFYCoKoGgns7h0BlZRhPyr646Ug/LkoVy7yZ2HUdISuYLuSaXQd9VRClSoHbGzqBf7PdiRyktsjnbOyhNweu/aAjKLx/ULo9di65IqEXSvdjyCWMEdnHTv3VTj2r/X7I2NTGd4p4VJuOivdxNEg+1Ho2pN5Rb2idFzNRxWIlXUm5TbsQlGt35ZlfZZXCdUgiqy5DhnRtKh/gbwQsi8EOvb6AxkSPNULlY1bDZS8Q+pSdq/XeZM7QWVZKAzJGyKrTTlcv96WVXHHeNX9vCteCbK9RDxQlwjiKvLzhh8WTIb8WHdeOFXCV64y+487cQyqTbi+16sgyJDkZFW+pmJKGAmjCCVxV+05o43Py9THGjYV1V4RFI5eLOFKzVl+VFTcvdj8KZs/2j8sIxBGQXBVgmfPF8qkYp6VP2drTj6cCy7kJAYb31QwEpdnI0orreV7v3DSX1YFAliUt8zqozX4+UnkxDXEC0kuiO+RLE7SQKBkAnkvrkBao8q2Ykezqo1VYElgYFc9NItsbOIjo+IVUI1HKcvCvbjildkTwVXYGloB6gl7akVG3OlltbxSJnJrMlSMCKY65gBUtGNnAVcUOM/Km5qlVfUgopCy+KIwHNQ7VfpIUCU/UdXufuWk7k3MC3nwL+j8zFDUrmj6ngsXMh2y+SMxSPBIDiBISZ3BWbXzctKS8Oc9XiEBaAHIE20Krt7I3Yu8HOIlFoqrL4vKyDXVZFQTXEXyVCbjiaVYccmvArwM7/7WwLPRXbSEF2D4krvPCr0aCuAwtGjQYvlIEoLj7JiXcvcuwCgCdv53U8RR/2dZmIZZsUeeBYuLVgy3xhz4qHQX5wS6LiQ48favhTCotnoBR5xnD96Dy2pEq3k1hSAf6UYm2H/QOIFQkAwXT4fAqKVnWfuysk6/gtSrkAaslBnEoEMO3CxK/tOFEKdO8bUYRBjyVdAELCFxmroYNcJoPZDRp0YIDbMkJSgAcdYggZXpstrUanGjibtlLg4Gn1VfzUFVyDwrjqVaIFDbhUe+rImHZclq52dxRdAhjnCl84xlb71PDAce1ExUoVVdUqBfbq0/ajGQZoplLQv9nqfHrL4RDrI7e1BrBsBtmpa4kubAtb8WANquJg2ETd44ksMiUvfs2cOpGxgp5CBLGe12zL6oFE/MrRqGKLdl1Zw2KOwKe8J8sKzJZ5pBKI9UCexN1libdl8eS9KhLvXLsNDe3b2rxHev+Sc5Czx7dnWVwdU4/uX0ofkqYsurVb5mHyr8LAtx8WH92FLzGpXA7ea0EzyYFNQXIW64EOCUrx/J0MDXIMpGXtVmxceL02WO2Mfn6iLQTmCRW3SxETiH8luDcTErdTjIo27LylmzalYohSKS6M/yoowVCodlQqrKhSwX3PZImiI7QSkYlwagxnut2Q91C9DdPimqr9b3iE5sBO/omxxCEYnnE/7VEF3WzfJVUD2jRY249JFakvIqjKWu2aL4slpeKRHUDiLuxOfUG5utEs1e8FYrzU1iohfgG9iJSj/Ef6A8ydNTVWoYkwXfZWXbK/GkheAfxtZmowQbUBlDxfl0IdIhXz+QQrWdoLpZ4kB9PoVwGVNOpLjqp2xB4x3PGO696oInII42wRylsmkeCDsn3SDZVV17dfdHsnT1iTWisE4+qF34TNihB3wZVAX2cR9vJEqJr6g3umkwVtblN2vKqZRl7HBZ6GcaCItS8Z6Xq5v+DqMyVPSKHBcSnPv7gRDqc4g7O/i23yc11ezJ+GzCcyZyeEGlV3CoGtfibAXmqvpU0BDaGs2elpWh7IOrKMJbtjXiMSwY2ug1R6vRxAtZrrzjQBrodjNBeH6LswTvkrG6xi5y3L3DJXX9AqaEX2h8ZVOq3lR93jQBot7IsrDv5VL1h+qYah7+7hq0vcgdiTIv6+WVIrVi07ukmT6FWbwWgoB9cBdPxJysHZbXNcFpyJiq6KJaDknDpaSH1QM0t2XhABJFfLxftTqRxBGPNftYiNAJ0S9EOnb4P0uhyTw41ugfzblNzWFB9DhP/ozBJ9s+2Z1QgP3A2om4W1bnrVnnLRtA3i8b2CoT0+yO9BPcLCnFIATFAdWCe0MOMOYF6oMDdffDwHvYis7MeWm9qd2pQrSquHOCNChSqv5D2C2qy2kGNdggLW4CJAr+b8vCvLPcqolZeUafw6eJTKd761iRcF3IcT5XdyCKBww8iia5zX7MN4LzRE6N3eT0pYtYmrqAKtJaXQ4SbpmIQrPqcjENnL9fOTu6OnopCBOqs+OOaybN4hP3LwS5mkD7syjFBlzB8K7WWdinm0kyVJ8JdSdfEZWgwKT2PFlt0h0GHw3j5R/JYvH9un0sKKgxhCxoqcaZqWvsMWnok/j8skpeJxC8TlkvMVINtT6naxafn90F1esUmbEaGxXSrKDNPApkSAyrxeQ1WrR5uUF6AVq6Rm5GKV5DrDZUAGqlC6GOZ+oO5Eg5sRViB2YS4vn0vGrQIvVdBXel6rjWPj3fSRPOhueLSF1W97HvRUrNPHEisDGwc5HhZZRPcoG7uUBv4Zc4Qmx0y0ydGlkOFCEizuldjfYGwXcjYRQ+dt4rpiWoJ35GaAfmtSzsc4AuqlDXbZh+dnpQStPEOXwGzL8Q48TljySp6oOo+KC0Y1Sq1csVPVb31e9pu+8SYv5WlmIIaXcygDoxR5Evd1tWzrFVDWxgWthum3amBDirVqBmULmQ5NRFjrUiNoHFGn3wUys+qf8HX4SJDwhwqbDn+Zea7KJXil7CeclSnG40LAvTbLV7USJimtVazD/IVxzyZKttvayU10kEW02a6dNtql05htHVZnrmTQAfm0YeveKsTdESZTdRXNXmdGEmGE7elpWj+KDWW1Vd39nMy7wXBCMOGlflB17Z2YHLH9pYtAqgpoQ0LzL4l2ZtPKApH5qDXJWoZQMYwErSgE1m2yUK5nk+P5E0flk1vSwr1ENz1DacwJ9EBrz4vyrPb+oWxrRAewTlPVziS9KuYq9mcldfB8CEWurTzOIUJwuRKSKPsNpNe8zqiMjX4YS+KWnPt2XhblXkWmqBcmqTalWeDa7I7Uj2y4UAp9W5RQb2RlBXulPimMuSIRFS2UZTzrhXP6ImSWwMRSI4NXW8jcsn3NvmbGRH9wtHKYJcUJfpvDqHZc6fop+iUXQsJ+YrJVwUtFZN8Ey4ka+a4ZqTmUZs5fMqbu4DZ94lG6BPGurL2r2o72YNwk2cWDx+WTijbiSTUWVSyt7760nToFHIy8m9rIlHhVF3ihyBc4ODhNkC7dZG51h3q9IpJ9AMt7VkEXPQ4Jy5tkcZOWvk9Ha/zuUZd6P5SFSw2ufj4fJ2RaarwHUhy2Ftbtm+wVES2DrNg5/21iqUX30T3eYjFIntPffWqpqEJ701zN64aMl77ZcssanBCjqtW3+k4A4H6RCZPf2QuaupqqmW9d7r4tmz0K+7K+qeiUFumqgGtJ2BOImrV1FEo7KrZ4+yu7iH1sqV9pqWjwZKcrZwIcepYx+Kos6jCsi6ouvmKGbSNRmN+rndvzXJK6RS3oDAqkv3OYrJchh7yKPNdr9yOpXuVuZxE2lePa2aUdNNAvlVvhDlwiUOhcEjstPTx0TvaHx2dcQ5/LYPa3hFTd3680nPzPJe9aXsOgx4x4vKp5aF8zpMUQvS2RSCnzjoi/rtSddLgLuX9fJKmcSvSL4VWlRHnTSLCKkhzeTm9UhsQndkdZPEKV/XLQoUpJauJj10mSTaLPOysk8Agigo+CpFmLVlpVY5qPQNHJULoQ79/VAOZXJASEYxPoU5DtRjxlu7Bs0meY/qAhHfDJU3nTLhVxV55VN6V7stq+ZlU/VLGmktiJ9nwh51rwE8Ssrhw5u2xTEW0B9rqa+X+GYG8uW2+xdEJ8XDOcjh1OEcIUcjlETY0tR7yAMToxXmNV+qaQzsa1mX5z0L3SpwGsPReMO8O+Z1B0tMGwO4EOP84uuBJJpfxU8Fii3sZmXFjaC8Y2dWY3KZlCslm54rataCjjYir4uvYeSEy8r9FHQ1SS02CPvuPrI7EMAmP9KFLFeXRA+kiZiM2gC6VRhaP87ZuzmSboxx4kqfwpht3HN2ElXz+6OcPal/pakstbKn2+uqmGacXH5ZKa+UR0NAqWtDmHlu53OylbxC7WXX7OMIBCv3g7Jqs58OymqGEIWFluaYNj9Oqa+NIV4p5/jS64EYIimaQag2DHY+KKv81YFCum8A0qtXuk/KqllwNilrJd+kXladB6KictJ9u6ay4CLCI1X5sk8ADpvq9831yRnJVhtxUvFp1kk12MFLGsWQg2QBY7XrMFiTkmPVge7XuTm+qLYpuOtlsvOaqD4ZwutTCmJPF7s/rcwtAmi6u6J3iHscZV2nfp8uk6jPPsfHg+4UFA2WFrvPo8t5m6pEyeaANLar+dj7da5M8yOcVetY1DjLYlm9Rn4QRJGM4UKEixLWIoQug4ak7KDrczPOMnNdHSO9CxFiqEvGHGq6y8x9N+UcZuYaOiBxbVpfpm97VZqw2qb852VFPCqMPjahqwvgg3++A06g4Um6IJFnzM1KQeBNpEi6Rq17fBuJupg7duhVpdA18PuFs17lIkiCRWmsbN7usUlg3bfuat9dyHJYhlu233TZRaVLTf3kfHq/jdxV2aJisloFSiee77fpIyFO7rfhObrd14sSknkBRjlx1Ydg6ELim75FGwnw7jEI/0JB7gd658DfZIeliiqp9D+HGpzEJfQWu0Ig3G4E1W7VdrwIT/ZKx5eFs5nDRlWLx1LrPhmngOVVu9A1yysJTtH7QIiq/jCxXGOHJVR/ht6aZtPwD6DdlbkGy/+e0Vtd23yC3rqGTdqkgS6fR8Cq6uTC84khvO1ClAuwO9JII58lfHbL9GI5nY1TSaMK9FK2S+y6gn83GxfGNeOj2TjVpRAQcyp1sHbCooaCCF92z+1lvbxSpv7nHP2ItXsdpiIXnlzGxfx6x9qDUpET1q4PYyAowIL9Pr2oJFtdE036YosXQh0i+JGhqWMMdVe3u8Ywr7ErKGBRFm531q42m64TZLVMgnLtDefBz3RTR8W1eluW1dlZ97qThIfnuncPzQH1SRxYa1xkeOSGW7krIT4PknYjuX2Mv08DU6L6/PfO2smeerOb0oJyp3Q9GG1Pdh3U0vVl4V5zt3KFt55D3Z1FxqY7wkE3W8/lOHf8A1GinusVEvWpAePTUHAbTEbhyO0X2dWwJmJWfZADj9UIlLMxM318RVQGQiTz/vbnlS7ZbdxidHOyEvUs9IEGuFa6EOTKRY60EvTpGiQX6hSFXSsj9xGk7R8qIsAWY8ehBexB1wxdMc6uO6AaD0Ip9+vmvpM+FwMCpgbULMkpe2v6HBc17+LLSnmlSJrbsQ/XSNLSLKYIvrI+CCDO2KOqO2kVrqrGtApCqvdG67MRzthuNda+LBz904biRDmLOPLMqQjbmskMIjWXhnbs9AdiQOWF3k4fLZJ6PPvsiqxOqUYdomasVZSKzx9eQbQu559dEXS3TZ/r0/feuvpbKoDrk6LSm/8HPUQcbihPAAA=', 'HIST_N10_PROTOCOL.json': 'H4sIANjCgmoC/6VUTW/bOBC991cMfN04lZS0TbMn15ZRo46dahVsFouCoKVRRIQiBZKy4xb97x1SlpItuntZXzyYjzfDN2/07RXAZI/GCq0m1zDZxBGbbzfLVXYzy7fZXyyJPrHtdsH28eTM51rHXWd96nJ1ny7Yh3S5zVK2Sf9kvja9T+d3+Wq7OWUXApUTlSiY0RJ9GS9L4agbl9Bq66a1LsBgK0XBvfu10bvOOoXWAj45VH4y4JVDA8iNFPS/XG1mayqynXQWDmgQHpU+qL5nLazThuAkqwS1YX3sGpzpMGQozXTbz8Aot22FengZ51KyVnKlsGTDaGhZQ3OxHXqXNg7LlyW7rnxAR54kol9wNbpEwkcsPV1/kwsgvkzOeuPtYFydjCQajGQwhuRkSE6G5Ish+SJJ6P9LaCj5jhoarh5/6vo+juKzwUpG62K0LkfrzWi9Ha13o3U1Wu8HK47G/i0XhhjLIpbFzPKGmNIHyw7C1UI9E/mSthfs7rk5sp12NevfQQCtRMZVyZzhQtGOwrP+sShT1MJh4ToTtLW4I13cZevfPuY366kj/fwOfhCojP6KCvZX55ev9+/OE6iPLZqWGwqSsmyvHJrc1zCrO1MEQBIjGsenO27xlBP/nINPJDiaDii0mN3mLE/vcyhqLB5bLZTr61ojGv9CVGXvvYZvgcJJyR2hu/6kSNiTnlq6HVTcCO0D2+VyNV/N1oT9Rz4k0Ogkcx/Ob7PB6Wq6i1pLUm7XH9zogdK/tREka6Cb2qESDwrmRBl34LghAcPyNoPoPIrejHjBz6rWEFaIDAG0YWZ30FMrSsLEJ1446FUAlrCnlRQtENFN58Jxgy/yvbN4mkVAY0MpqooOWBVox3fRWQuvC+GOvoPSCmmPRLFEaAkcC8l9jxOnUOiGViksfcII4Xvgu6ouSC6FVqVnveKNkMdnzgdun2+EnIvtzWy1CV+79H42H3imSJ7e3K5nefqr2Knqv1KC32+Ofb6cBOeXX60wrKHGXrG0lmmJRuzpnYH2sJpxl5P/u4V/I/ujlg0RagxdVfjyFkaHTzGhyiNUJHsY2AtANnD+6vsP84Ef20oGAAA=', 'HIST_DEEP_N3_ALL.csv': '...', 'HIST_DEEP_N3_DELTA.csv': 'H4sIANjCgmoC/7Wa2VIcVxKG7+dZioqTefZLLKEJImTksbDDvupgLGwTgSVCMNvbz/efaiRkIbBNVVtLq7pNZ+fyL3nqn/9688v5zfTm7Obsmr+vfzp/e/b+4t10c/ae67ufr95Pb6ffzs/e7t6cX96c7W6u3u+urqafLnreXb778PTXCz29end9cXPx7/Pd9fn5m+vp/L9nP93sri9+efvz5cXV7mr69d3lb7ufz367uPzf8vxqen/2n93V7vJmF65C3l+8/effPPCYnh99P36HmX/ZFKcy5xTvPDIv5daCxRJKCTlVs8l8Tr2VHEMJoZj1xv8ZZs/8rNfPvj3+5vT4+6PdyavT3fHJi6Nvj05Ojw9f7l4cfn388sdpenF2eX2+/HlfFJmfZTZbmtJcc3A+vQd9dvPJ6py912QWeo+xe1zvg/Xt+1xiufOoiqGbx55qqq2nlidLc2oeemy5kauWVvzyri8f5+p3S5C4Eq2VHmruOUbelGbz2vlnqMYbbMUQFIHPrd3NQqiTzTnUkHmeWk2xNyMIL8Urv1vK5Ga9WtiIIdytRFEj1tJzt9R7raW5j8SUSDs0GjWXHNbLg6sLSUe/+2gaht4zjd+6hdw90Q9z8pBS4QJJoEZPi+HF8cnhy+n5q68Pj092r1493x39cPjs9OOAjid8emMCY4heciltOsjMSvVcYqBfLfJWXekh8W5eMRonTaavtEFc+sppbk5XeM3FXH9PB2BJrTHHwpxQOScnVmaPpRMjNXRerptla49mfGqKNabQQK8yHRgBpEwXE0RkjkPURDNNVJPZB1mi2WZBjQHPtFdTQmhiJtgpX5i7W84t8gKxhdYm49qAHICW6afQm0W1oE4kvM4vEJ7igL80O1lzN6vAXI9JCQ2ULrdUR1FD/xjT0Te7k7h7fXr41fHL49Mfd/7Vd8//fnT6evfDLr9+dnRy+O3xq9ejAf9MbDbm0KMVRszAn86Hxukg8VIrVYGllEsvhJaadZDCcgsJmMiT6yusnq49NFg12DBaaiVa7OosLtUS6STqWJzQZ4chaT73FrKLPVYL6fTo629eHp4e3Y8RlCzXQqUIhwAoEAgRagPCk5O2oPAytAKL1WZNqA6MrAYQX4puQYrEp5uDp2nhcJAC+hBu0fHAWSGOPLfSjMmFeiOqY8Xm/0Jwe8Cgu2Fbr6lDQKayDs6BeGl5VzACjMqbIL0u6qk9bR/dHjmQQ1YMTnb0AOlU19FvxNIEG+CsxFOh9mTWmNnyVH3yR4LbAwiNRYchFcUAbfQhQII+UTwAWISyyTDwgYDInWu9ed4AQu4P8xZLakrwk1skaWVgCcDvUHdjLGI2YUlDY6LCmFlwuvUxt3XDFO5BBRKQAq0QxEIMAhVCQ2NR0CCqB1RgCroR+EmMyCq9NwI6PXp9uvtH+ogjJKvlbN0ggcol2JpkMQswNyiSoYdMUqqkmiOLOrIYbRLXyNbnEalR6oziFg2EFvmzpOmgzUWDGWEq5FAW7EdMTGqxOu2YBMCbpGhkKNA7aIiEDEUZlqyXmLsiFQQRKNLok4tKQzLYIpeAZt2maGOW2lyRpxARsicH/uMleoqpLPQU2KGWQYcxeTJxoQCxcZsM7XUhQsHV0A03hy4MM8UhxooAE1dmx3c56iyA/h6Hll4TE34f1t5dNBUEZnaNOpPmVBLAJFHgQO6LdkRkAFGyGLFukaJl7Gkb0Mi6vj+Fqxp7oispaMQNgVOVSNiaQeugV21tHZH66sWL42d6o2L6OPgHesbglNiWB4qmS0IULEZwEVAM6GgIiXwlWZEmcsxxBQlxT0xLI8l3otFteUg1JFoepeXdGxHEoRp6kNzPlo1vELdJ0iIVAGTrNK2Px8LFpVCwXiCOLI8zWUTaFBEe2VRo2wTk+1GDIjzd2lWhY2DOcTkImJC0O8BFgFloaDJ0e22bkPaaIEtttrvrJASWYMgYqjp8hpxqqppHDDW9Vtac/s8CW4hNMdQPvSQhFemv0lF1ESoJgiZK3CksJELTo/dQg0+X75/Fs0AAQ/6hk8CAPixhb8MLEg/ckoqoH5ZL9DzRhAQarFG6R5wEIUjpBq37MqFIAvQs8I7axySsIBVE4hEXbxSQtzVWDQ9bCMi95AwikIxaisAAtpMMiaAByiALDJAqXWoPdwFI9A2ztSBCwn4Ci5Qo4lfjgAS4jsZOkEkVWA33kOBDQCKEmjLUsmFce9tgfH9r6piKOx5rkJCZvZx1oYVoVb4huiqYELwR9PAN41rQISHlZNpNflkfJrU5KgalkJpGTp1OA1UhvJxCWBUbHrIJqFlACTTXkgpIGCsHJjJTXlRvRBWgRklqp8BMLF3XfQV8eMgeGLIbmRTBSdemYwEJ9K2hVECG3MYaB2Q3LZYQXE/srXsOCBKy6NMV+YHXmVbS5op0JfX6FBlHmT76qErVtCdZgntOCPhhv9vUk4tKGYDL3jrIVHNF5kHQTFiNHrXYWzEZy8/qdxf1EdjmJQYaxkfrIt/awEJpbeQdzj0Bjz34inHc+uf0ybJc4lJhAIkdRW1WQxb3YnkLEI1u6m4oubRiJPtAPl2Zh7HcDBZqZ6Q7kaA6msJL8c41WL+uF8p+T94+OcjyMqIUuGh7At4wvKaKGcZVqgl4SZFpXi+QpdDU4O5jtGoAQJppKQjpJ1grj23JJxfjU/vksbOEOJMitUYFQ8AvLVblDKkKn48vAmQSZomZ19IpSv1XoktPnObHzhIczQz854xqzphlH46EHuk6BFMFZfThhko3t2C8U9zQt0vXvqOkw/hEIkg6AJRpLAl12IbGqL2BczPSKGtrXhr6smbtfP+yFnr8MIHEOPwEb0JBxNCqJFpEkJE+goP1IYegwIrhktAgoBJ0i0Daqoh7JAB0XfWpXSTgo+9NwjHIN7UgLNRQgoq5yXUXMogiWYXiHztU0I4cW4RTTLggt559xCfQRJNUZgBSKwMzoTT0ZkYF4w5K2azN9nDBKCatb5EajGcfLk6YRePBpwaaxLxgPtoETYA7z0WSZM2xfMgVJD1BkbWijZdkxwFiuzOFY22OXwJmtYZjEJKOz4M1LVufIo/+xAGDNBDgTvMzEG6DgbQjB0e4pAPS6tJu1L3msYdCm9c1q/qASVCZia2QwNCXjkrCEViZ5GUKTNWz6Qy+hnHPhZtryRJXxJGHzMJoNgd5dXAlBYkVEKAkHfdJvVSDM4YbNW2tsoYDBiPUNRHlAd8QtHmJJsxFQEhSDSXsWpgxvt6Ndgs6ziLX2r/GIFoFj7dAli+bCG2CtUsMWkWncWA/Xurag8IN6LJMsGJ+NIkkUqTOUFlp2zfjHmvotubV5Jkto5NHIjUnVTeaECfQvQQYUnK8YtSxr6U1Rvn+Q4eiU5AEgAAdkY/TyWqfVdjWdcoAIqLsI6Wp4Azp6jrbr3GbiJZtg+P2MH1kS2fj0oyojq57UPAbJqcKntTiMtIDEs3W0W73Hjq45gCXBYublviLEQRqdaolOCbOsTDOlVI1R0wR1pPdz0OHDlSjiTjVwmXsFLU5QnqoqC2h3CK4oHt6dD6EKYI9dDSyTUgLSCiYYpJh6pAmbDLdDgNZ6jaHWFqQN4MgQDTc6liENF8VI+47e+Cn9yyaLL54Dl/27YBpEevT+FkvjVvv1HGNX7AE18sm+Vq+svnASVhT98toeSx44JruboAQYMnlSDfrfTBXwxDw9rRKTF84iMChtLEIWja1oQ6DggWJ2s+AWqhZ+RMUhu6EqtkIn3lYQwh94RwCcwIGhNuQTIv/MER/pKH5p8kNAGNFuyFCAkyBs42ytJcUHcf24bCmdJOiQD5gMovO2S3CQYpcRyLjfqjW1eorKIovnEUMPu461ErjEXsfrgROiaXjCXRXk2lDU3Q/SgzdtLWh11YREfcfRygmWjvfLgZ0HLN4EvwcLaXj94jLHeDhWhAE4KPpO6yKCveeSch7MGUtLacAJGTAujkVQwp2yZxax32QcJEZkp+Gy6GXbVrrdneRPh65Be1L9jGVXiuOk3IT8chs08IA2AJIUupr0PEjHiQj6WphEBOOtg4PUoB2PDC6AGdUmzxIZVYdtah7RtumYS3eQ3dutBQbRIPOt+HeqKnuXxM71qJ72HTPRNShL4Oh29bWWV08bDsqGiYNj637r3Tk5brHKlFA6oh5q9rB0YxDwxQtMBE4a2DEY3bDa5SR1S1ButEvCimYBMKF/YDUUGy4jSha1L4Q0f+km9X+wOEEReNj6W5J4EbR0sB5QIHy4ba1RtHKTEGkzJcBTdBZOMm+KlY8aC/QV5BwiF3nSMnH9hmZUGUvs/aq41BFPVZ1mgEf8U7fsNf2UgK1iSQPumkOV5PzYirxYA3mQV+gEcc6lhQ3piRXSVaR+l8N7f82zOvXazEAAA==', 'HIST_FF1_PRIMARY.csv': 'H4sIANjCgmoC/42STW/bMAyG7/stXsFviscgSIAAaRekvQfe6hQ7rA0c/3+MtvKFHZboIFsiRfIh3747NIf+62fXvLdDe+yG5vir+2z731/N0PYf3bDbH/pmOPS7P137Of0ch/dmf77Yny5mm3rO73jc7Nphtw2ol5dTmr7NX2fPi2a9elnMts1y9TJbNz+Wy9V8NVvv3havbw08AWjuGqBRQgvXNRomo5EXNw5HcDSrhlyn0M/rzX/jGqgGopwDjwbEQipUwMmDTbV6e2Hx6z5dAgrnU5VSoBC76ZR7C48weT7OdwTlX6YsiKMIEQT4FSnD3uVxMynCXosOFmH0LAwRGK36qTKbZO0BJO6VhNAtnBM7CC1dp4z4EIh6Bie9AOSoKK77DQHeJXDMxpdTtOyDwFhklidigNWNmQCLQAlBKXYGUBMLJkxsRKoJaf4IAeQIrfiVIE8pN4twLcHV5xLwHgJwXIZaJ0tPGYvAChumiL37Xh0zj6GApRUiKGdhKcl0CU4NhjrL5Dpl5odQFBCyEzfTyA5pSokkJ6HGNyx8n4UT5Eaj9XV21wRV3ZFZuA6AXZSFYtRapjvJyinlVCRVxQrZ0inxXyh/SvppBAAA', 'HIST_FF1_PAIRED_STATS.csv': 'H4sIANjCgmoC/5VY205cVwx977cctrZv2/YjIhMJCUKaRFXfRpRMW6QUEIyqfn7XPrRzSzuzOQ9hYjH28WV5LfP1dn37slpPL3erh9vn+8fp6fnxl9V09/iwfr59WU8P0x+r24fl19W39e1y/fS8fHqa7u7Tlt8eNx9/v+8fnx5f7tf3f66WL6vV15dp9dft3Xr5cv/bw6/f7p+WTz+8W/y0XPx0fjVtPlxdflicf5o+0dmnOtkURXL38SGLTbXUxnbEP1+8BjhzLa51+9CgqZ6OIZsYpfr+G46ZTse4+Hx+vXiNUovv1aHxoOlIta6vPm5a0V+Pd0qgk8GXSc0mSl5FGa3QGi2rNAvO1uyk820fWgkO/PQSrhTmgu+ndxMCBQUpNxGtrMcKM3vdVJ6tsNh+naWW9FqNuQpzY4eNslilVPds7uLkp2LsVL6VtluYuTINjuE+mpGTJarXSLpzEkWemtvKvL/8AJ83799fXlyeXy2/LD5/OYSBFm6Kl01q/XEdsgxF2DSgofBJyV5D5mfQVEeibBtCRcRZ0v75fhu2jQTa6coZcRGP7F858p+jXnfnXwsGMFu4vXaZJynVqgVlQ19RlcDm6eMmZoCFMAELx5uwD4EmWDJaqeprC1GGpiUru7YIwoDOJi4WxoCxUQj1Rp/OYVN/6TBrsR0TmKKgnSqapNIqI4ZYkegJIKl0qXE6xG7lvVA1/zfEnEiWGt2oZFFJgqczKyQVSzXJTYC9PAzy7ub6/PLD8ubm3XLx8/nFf6KCURaRJLZIHrLYYJBtW6KkoO941/QGdI9YRlPZBYY5NXL32vcUD5oGA+0BQwphTL2yCWv1FsO2U9EOAMMAmSeG2o2tdcBUhkuGTyx3sCjwYtaSGwYttBtGAmxbowXkY4GBxlRndHiALLwxK1CI4RWacaXJGDDB5NlAcw4gg3GOCJC+ZkiF9xkzTWoNJsx4OlbJDBpHoiCqzn3S1IYC7QMnOi1hkTeiUEACwAmQRkgj5KMoVQdOYDUyoN/wg+t3q/HL4vrj1fmXxTHoKEtn2WyABdU2arLhSLv4CQERhksnxU63Q6bxpHZBhHVsDd+CsgHf6qBtPNgBkLhiAiihJhKjrsO20/H2oQQOgHroU62YxQ4lgnABtJIgy9I79SgTsoOmg0TyHOnVAZrgonbxE6kYbu1dadBFndtEwHWYyRlzUhvmFLjDDnIbzWWXgyC0oOQw6Fbxxq8gw36RjEB9WgPjdDxRNA4iA9eBTQYD7QMKEAlUHdU3bCPvgIIGMyCZkA+Q1vHUBP3B74CGhOz/FsQQrAga1STQncTqHLLYW6NtO5agHfQGXiCOg3zQ9Ob8tghjnAG9cND7LRUTOWx7a9ADpIEhhHFtQKIIxaBpOOYB2jCf/QyAdID8eVV6aJph7Wbin9bRppAU2feJ4laJ8S4egK4VHDrkDF3JTp1IoLHV4BVA7jqTeYYm4B7YxDhcgJJ8Y2Z7+k+gH4nguCNOZzJDo7BWDPli+arO4DOD6ASNsWL5vKV/hyDEgY5tIZgCI9ZZWVRkiJUCPYZOYV/1FCEFCMIAT0DzfxdvDtLl5vJHPcQdEGzoeANDY3ngbByzDATYlRrYgJm4GKHQRWLQNJLFpjsYYMFKwCA4uKL6kGUgwm43cK1i3TlORUww54DhWIBd3Bh+SxOjC4yoBPhDcAdDQWDdVTccFzLNfILhxuWR/ayjo104gEotEFUqjnuoWtMu7agkdhpkSnaWkE7ulqXf2PANrIBmWE9nsMVH/+OAYE5TMZydvAXHIuFmCbBtBu4kLLdEXr326HSgTn4ywG4DZjGgDuh57dDvBxfg3afT+x8fcGMSukA4jxwXS7/6vCkOzW2UvwEu/4poohMAAA=='}
EMBEDDED_ASSET_MANIFEST = {'HIST_N10_ALL.csv': {'sha256': '1cda3ef91bf9c602c25259e595f9c05261fea4d7dfda53645675736aab832f3e', 'bytes': 20264}, 'HIST_N10_PROTOCOL.json': {'sha256': 'b12f7404fc6546e04d20bc14eab3a9ca04373d7c7cff0e354f62bc742673478c', 'bytes': 1610}, 'HIST_DEEP_N3_ALL.csv': {'sha256': '276b646805820f014b6321a056f1dc133ad6bb893377e8656e09d40f55800a2c', 'bytes': 102424}, 'HIST_DEEP_N3_DELTA.csv': {'sha256': '687bd400462419b505aa4b57b7426d6d539c29532d6a4bba0c84af6332c9504d', 'bytes': 12651}, 'HIST_FF1_PRIMARY.csv': {'sha256': '1c14c567739323cc86da628a78d95a71782dbb76f65e3156820f8978a684c5d2', 'bytes': 1129}, 'HIST_FF1_PAIRED_STATS.csv': {'sha256': '38926ff712a208cd497a4064b38d9f30d8607bcb71dc63883de7834306f06104', 'bytes': 5026}}


# Additional independent historical system-reference evidence (v8.4).
_EXTRA_ASSETS_B64_GZ = {'HIST_XGB_N3_ALL_BUDGETS_AGG.csv': '...', 'HIST_XGB_LABEL_CURVE_SEED82_PRIMARY.csv': 'H4sIAFPFgmoC/5VUW2tbRxB+z29ZxNwvjybEYDBJSJO+CjVWEkOIjaT+/367wsVq+1CJg6Qd5szOd5k57vcP448/H77vT+Pw9HO/Pe2+j4fdaXdE4Ph1/2t3eHwap90BCdtvz4dx+nHYH388/XwYJ5xm5Pmw//p4fHz6Nb4xgoiNE/7jed4+7w9bJqJx83F83O5O209Nb0qGzNh73d5+ub/fvv3y6fd34/bu/c39+HB7e/f27uZ++/ndb58HbYh86CaLzavcIl1aEdcS6lA3jvU5pwbOaVXWZoxQlyR3qycJVRhC7iSWXdxipJ0D9cyHc4ymNBuW5jhvLmrxvKCRThaqyUq0qpWZC2ImodVpE5tfg002ldF4IsqzmeZFFhyuAci6PudcjZa0QHXTWuBCoyhE0af0fDGk2hicWFICG/gBFmua2JKG8cK6uSili6lgvMdmYty++Cywx9xcBeqAbmKbWl4DTgOcagZwSYj4FKAJTaX0Wbk85xalQyyl8IWkQbylJqGrJhaEUluaujS0O5yHuVoO14UO3wqTKNBd1PIpE5QRiwDads5zSBNQ3cG8uJG8uPIadCSszAIGmYFoQgEiYUJxX1k4KHGmMzwnC1fDdYHOoBA6mLjAPSzmVDm9aPAi5B8OapoCbheH0UdsLmr1gtUOnTsaLoX7Z8iQUCwmUomfF0P+f1gM59MSvTAAsN3qUbrB/4shz46Etrg0g5qg5rwcf9BcEczFkstERkYwJcysmESgAifDIQE0g9kFyhWwXZSys711zi3mLSDVwuZR8y30IZoZfxvyGnDmpJP5KNjfltHTHcsApL525L+Vg3CSmARx3J8zhIlVLAX4KJEI/mFjf60cF5bJfyo31wq7ilZON68JBHKfYfVXdrwGm06Sa64DhkY0VQJSHP+JzUxZgAKvrZkENsawzzUGG/LMKqjRJli6mYUmh/esiC08sQE+ymLDxOaiVud5MaVgxGEQrNla0xYwN8yBQZz8yZu/ALWFbA94BgAA', 'HIST_XGB_OPERATIONAL_BENCHMARK.csv': 'H4sIAFPFgmoC/22MzU7DMBCE732KPMA2cWznx0LiAKiAKBJCrcTNcpJVEjV2ItshDU+Py4UicRnNzn4zbnUeNXTKNouyCBqVmy1qNF66epwQKuXrTvrOjnPbTbMHI6s0yFVicULlHQzKo6nX8Jcam14Zqf+Ek8guyW9TTqpFJ13Alfkvd74BPTY4BBd2ZLV6dDDbQX6Otaqk678QPJ791b057J4fdvLj8U5SQl7g/u0Ivdlq1KNdo+P7PrkUou1tdNhtA3lxAY60sm1vbiI818PcoIua3p0Sg34Z7Smp7Lg4tIlVS/R0eN1Hk7KuNy0wCowQyEgQBizmZcZTwYUQJREsg5TERUpFnqeBIAUVDHhJY0rzjAWgYCIvGRSxSFkhMkoY54ITCoLnLC2B/gxfhGy+AUzAYUatAQAA', 'HIST_SYSTEM_REFERENCE_XGB_VS_DEEP_N3_PRIMARY.csv': 'H4sIAFPFgmoC/22V206bRxSF7/MsLtrnw2UqQhSpFyhKpd5ZbnAqJAgIO6369l0zYyNSGCT8M55/5pu119r8+ePmr/1xc/j3cNzfb75vjo9P2/v9bj0cjjebb+eJb6eJx6f919vD7cP3Nf3+en1eb3fH7eem9dfdwz/b8cLXh/vH3dPtAav/3t3d3rwTItp8ufp0ebX94+OvG93QhWYGJ1daz4EpYra2EBUJi9Yc64jcQyoomim85hSxFwcVluINphRMd0m2c6YIpdmYUcy5SwRriI3tykxD07yDXFU3V7u7w/6dvwa0KC7P0FpjwrCVz8NxqCTlAjQxCmXGxpSRG7vA5dSFcT12sv0v5BMwjKVIiSqdxoYd7PPSFhxNMQi7Ldk0g7oBeiJkeo3oFQAjodIxbNJoYXsW8UmfZxErowuH4dtkLI0LUazTUm5oUfoMCVW7GbcLj1kXUODODP2hLmoxlQ1iaTItrEs7QcqEvPzw4Xp7+fv737afaXImNXFTozJzDCKhIHXsQhA0INvCTA6Jco0E26nWymHmxpj1mPfpxlvjdqRCMhd2aYAG3mmP8Fn+VHyHgyOxpejmy9OPNxh5MZoMOeG2NWa14ZGSYa9GtX3qM/wIf5YAKb2HhRYlG1TKwmJYTHw6EjJqG4kJ1i11HVUh6rJE7aeUWdDQIEWbNqZeUP5cblzX4cpcGGVSUtwyal29JoeLuJCdMpTqmU2j8A2qyTV8kJNkLGSoU60WE6RhWbBC65SYolrCC9IwMoBMFpu/wZaauJI8jykfGYyOEzvElasXIuwf2SP/EGYBCm6BpA6L4JmXUt2OZQgufvGqsCM1I96KqelMqFmGH+jLni+j8grQh/vJIMMYK7nIDGyEvWAttJJYOJO2HbHI8rE0L1ogXiIHMCwAzlmBLXq8DCytWLI6exjOQv5rZSUGd6GPoC2V5susvBGWNkQNADJlJp5IAikJocANkGizhY/2aYGPNEyd8z/ELFDjaeS3VxQQXkMhEpkPXq1n9EosslYZkV9ThfaLk9Az/YUR38gLrAFDoUus3qMnTKQ2h2HQ0s4iBxCseoQ7cfTZk614RktAhhudcenJkAqZTWSZ/ISEaoC9sDLP4NiyoSanB/+E+b+aJ6xyFvJZSoQD/zyyoaLUScnQCDQA7+RxsTwxQpdElGFiBChnj5o1RxdCw8btVzcqNH70sQ60L17pwuVhFhv/eUThzAn5H8rIoVR3BwAA'}
_EXTRA_ASSET_MANIFEST = {'HIST_XGB_N3_ALL_BUDGETS_AGG.csv': {'sha256': '57a3e6296145d776f80ea3625760325e8215033cab1c2551e05336d5adb24f28', 'bytes': 51296}, 'HIST_XGB_LABEL_CURVE_SEED82_PRIMARY.csv': {'sha256': '791ea65adc53d4b9b12b785d6142e4fce074a1c2186cd48633d85ae5346063b8', 'bytes': 1656}, 'HIST_XGB_OPERATIONAL_BENCHMARK.csv': {'sha256': '9e3c7e208fb4b012ac585a9a3cf93bed918083089a1399ec97674edb01a301d6', 'bytes': 429}, 'HIST_SYSTEM_REFERENCE_XGB_VS_DEEP_N3_PRIMARY.csv': {'sha256': '6e218e7d91bad37f47aa07826c5860899984016d99c0c8ae557e02f053a37d4c', 'bytes': 1911}}
EMBEDDED_ASSETS_B64_GZ.update(_EXTRA_ASSETS_B64_GZ)
EMBEDDED_ASSET_MANIFEST.update(_EXTRA_ASSET_MANIFEST)

HIST = ROOT/"historical_evidence"
HIST.mkdir(parents=True, exist_ok=True)

for name, payload in EMBEDDED_ASSETS_B64_GZ.items():
    raw = gzip.decompress(base64.b64decode(payload.encode("ascii")))
    got = hashlib.sha256(raw).hexdigest()
    exp = EMBEDDED_ASSET_MANIFEST[name]["sha256"]
    if got != exp:
        raise RuntimeError(f"Embedded historical evidence hash mismatch: {name}")
    p = HIST/name
    if (not p.exists()) or hashlib.sha256(p.read_bytes()).hexdigest() != exp:
        p.write_bytes(raw)

atomic_json(AUDIT/"HISTORICAL_EVIDENCE_MANIFEST.json", EMBEDDED_ASSET_MANIFEST)
print({"HISTORICAL_EVIDENCE":"PASS","files":len(EMBEDDED_ASSET_MANIFEST)})


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 01 — Eingaben robust auflösen: Data Freeze, HTML-DAPT, DOM-SSL und Basismodelle
SEARCH_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("/mnt/data")
FREEZE_ID = "PHRESHPHISH_FINAL_DATA_FREEZE_v3_HF_SHARDED_DOM"
ROLE_REL = {
    "SUP": Path("roles/supervised_pool"),
    "DEV": Path("roles/development"),
    "CAL": Path("roles/fpr_calibration_benign"),
    "SSL": Path("roles/ssl_pool_200k"),
    "FINAL": Path("roles/final_test"),
}

# --- corrected metadata root ---
markers = []
for p in SEARCH_ROOT.rglob("FINAL_DATA_FREEZE_COMPLETE.json"):
    try:
        o = json.loads(p.read_text())
    except Exception:
        continue
    if o.get("status") != "COMPLETE" or o.get("freeze_id") != FREEZE_ID:
        continue
    mr = p.parent/"manifests"
    score = 0; reasons = []
    if "freezeresume" in str(p).lower():
        score += 100; reasons.append("freezeresume")
    if (mr/"train_role_manifest_PRIVATE_WITH_LABELS.parquet").exists():
        score += 50; reasons.append("private_manifest")
    if (mr/"final_test_exact_ood_flags_SEALED.parquet").exists():
        score += 20; reasons.append("ood_flags")
    markers.append({"path":p,"obj":o,"score":score,"reasons":reasons})
if not markers:
    raise RuntimeError("Kein korrigierter Data-Freeze-Marker gefunden.")
markers.sort(key=lambda x:(-x["score"],len(str(x["path"])),str(x["path"])))
FREEZE_FILE = markers[0]["path"]
FREEZE_INFO = markers[0]["obj"]
META_ROOT = FREEZE_FILE.parent
MANIFEST_ROOT = META_ROOT/"manifests"
if not (MANIFEST_ROOT/"train_role_manifest_PRIVATE_WITH_LABELS.parquet").exists():
    raise RuntimeError("Privates Labelmanifest fehlt.")

# --- physical role root ---
roots = []
for r in SEARCH_ROOT.rglob("roles"):
    if not r.is_dir(): continue
    root = r.parent
    if not all((root/v).exists() for v in ROLE_REL.values()): continue
    score = 100 if "finaldatafreeze" in str(root).lower() else 0
    if root.name == "phreshphish_FINAL_DATA_FREEZE_v3": score += 50
    roots.append((score, root))
if not roots:
    raise RuntimeError("Kein physischer Data-Freeze-Root gefunden.")
roots.sort(key=lambda x:(-x[0],len(str(x[1])),str(x[1])))
DATA_ROOT = roots[0][1]
ROLE_DIR = {k:DATA_ROOT/v for k,v in ROLE_REL.items()}

def nrows(d):
    fs = sorted(Path(d).glob("*.parquet"))
    if not fs: raise RuntimeError(f"Keine Parquet-Dateien: {d}")
    return sum(pq.ParquetFile(p).metadata.num_rows for p in fs)

COUNTS = {k:nrows(v) for k,v in ROLE_DIR.items()}
EXPECTED = {"SUP":4000,"DEV":20000,"CAL":50000,"SSL":200000,"FINAL":168060}
if COUNTS != EXPECTED:
    raise RuntimeError(f"Unerwartete Rollenanzahlen: {COUNTS}")

# --- historical v4.3 assets: HTML DAPT + DOM SSL ---
old = []
for p in SEARCH_ROOT.rglob("phreshphish_FINAL_DEEP_TRIMODAL_SSL_v4_3_LABELSET_ALIGNMENT_FIX"):
    if not p.is_dir(): continue
    score = sum(x.exists() for x in [
        p/"checkpoints/R1_DAPT_TEXT/COMPLETE.json",
        p/"checkpoints/DOM_MASKED_SSL/encoder.pt",
        p/"audit/DOM_VOCAB.json",
    ])
    if score: old.append((score,p))
if not old:
    # fallback: direct R1 checkpoint search, then infer common old root
    hits = [p for p in SEARCH_ROOT.rglob("R1_DAPT_TEXT") if p.is_dir() and (p/"config.json").exists()]
    for p in hits:
        candidate = p.parent.parent
        score = sum(x.exists() for x in [
            candidate/"checkpoints/R1_DAPT_TEXT/COMPLETE.json",
            candidate/"checkpoints/DOM_MASKED_SSL/encoder.pt",
            candidate/"audit/DOM_VOCAB.json",
        ])
        if score: old.append((score,candidate))
if not old:
    raise RuntimeError("Historischer v4.3 HTML-DAPT/DOM-Assetroot fehlt.")
old.sort(key=lambda x:(-x[0],len(str(x[1]))))
OLD_ROOT = old[0][1]
TEXT_DAPT_SRC = str(OLD_ROOT/"checkpoints/R1_DAPT_TEXT")
DOM_ENCODER_PATH = OLD_ROOT/"checkpoints/DOM_MASKED_SSL/encoder.pt"
DOM_VOCAB_PATH = OLD_ROOT/"audit/DOM_VOCAB.json"
for p in [Path(TEXT_DAPT_SRC)/"config.json", DOM_ENCODER_PATH, DOM_VOCAB_PATH]:
    if not p.exists(): raise RuntimeError(f"Fehlendes historisches Asset: {p}")

# --- base model resolution ---
def resolve_base_model(model_id, exclude_terms=()):
    needle = model_id.split("/")[-1].lower()
    cands = []
    for p in SEARCH_ROOT.rglob("config.json"):
        s = str(p.parent).lower()
        if needle in s and not any(t.lower() in s for t in exclude_terms):
            score = 0
            if needle in p.parent.name.lower(): score += 30
            if model_id=="roberta-base" and "carmengeiss" in s: score += 100
            cands.append((score,p.parent))
    if cands:
        cands.sort(key=lambda x:(-x[0],len(str(x[1])),str(x[1])))
        return str(cands[0][1])
    return model_id

URL_BASE_SRC = resolve_base_model("bert-base-uncased", ("url_dapt","r1_dapt_text","r2","r3"))
TEXT_BASE_SRC = resolve_base_model("roberta-base", ("r1_dapt_text","r2","r3"))

def hf_kwargs(src, model_id):
    if str(src)==model_id and model_id=="bert-base-uncased":
        return {"revision":BERT_REVISION}
    return {}

INPUTS = {
    "data_root":str(DATA_ROOT),
    "meta_root":str(META_ROOT),
    "old_root":str(OLD_ROOT),
    "counts":COUNTS,
    "url_base_src":URL_BASE_SRC,
    "text_base_src":TEXT_BASE_SRC,
    "text_dapt_src":TEXT_DAPT_SRC,
    "dom_encoder":str(DOM_ENCODER_PATH),
    "dom_vocab":str(DOM_VOCAB_PATH),
    "marker_candidates":[{"path":str(x["path"]),"score":x["score"],"reasons":x["reasons"]} for x in markers],
}
atomic_json(AUDIT/"INPUT_RESOLUTION.json",INPUTS)
print(json.dumps(INPUTS,indent=2,ensure_ascii=False))


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 02 — Datenprotokoll, Rollenintegrität und technische CAL/FINAL-Sperre
FINAL_UNLOCKED = False

def y01(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s).astype(int).to_numpy()
    z = s.astype(str).str.lower().str.strip()
    mapping = {
        "benign":0,"legitimate":0,"legit":0,"0":0,"false":0,
        "phish":1,"phishing":1,"malicious":1,"1":1,"true":1,
    }
    y = z.map(mapping)
    if y.isna().any():
        raise RuntimeError(f"Unbekannte Labels: {sorted(z[y.isna()].unique())[:10]}")
    return y.astype(int).to_numpy()

def _guard_role(key):
    if key in {"CAL","FINAL"} and not FINAL_UNLOCKED:
        raise RuntimeError(f"{key} ist bis zum DEV-only ARCHITECTURE_FREEZE gesperrt.")

def read_role(key, cols):
    _guard_role(key)
    out=[]
    for p in sorted(ROLE_DIR[key].glob("*.parquet")):
        names=set(pq.ParquetFile(p).schema_arrow.names)
        missing=[x for x in cols if x not in names]
        if missing: raise RuntimeError(f"{p}: missing {missing}")
        out.append(pd.read_parquet(p,columns=cols))
    z=pd.concat(out,ignore_index=True)
    if len(z)!=COUNTS[key]: raise RuntimeError(f"Role row mismatch {key}")
    return z

# Physical-row layouts can be inspected without reading data.
ROLE_LAYOUT={}
for key in ["CAL","FINAL"]:
    layout=[]; off=0
    for p in sorted(ROLE_DIR[key].glob("*.parquet")):
        n=pq.ParquetFile(p).metadata.num_rows
        layout.append((p,off,off+n)); off+=n
    ROLE_LAYOUT[key]=layout

def read_role_slice(key,start,end,cols):
    _guard_role(key)
    parts=[]
    for p,a,b in ROLE_LAYOUT[key]:
        lo=max(start,a); hi=min(end,b)
        if lo>=hi: continue
        tab=pq.ParquetFile(p).read(columns=cols).slice(lo-a,hi-lo)
        parts.append(tab.to_pandas())
    z=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=cols)
    if len(z)!=(end-start): raise RuntimeError(f"Slice mismatch {key} {start}:{end}")
    return z

# Hidden labels for SSL are isolated from physical SSL files.
priv=pd.read_parquet(
    MANIFEST_ROOT/"train_role_manifest_PRIVATE_WITH_LABELS.parquet",
    columns=["sha256","label","role"]
)
priv["sha256"]=priv.sha256.astype(str).str.lower()
SSL_META=priv[priv.role.eq("SSL_POOL")][["sha256","label"]].copy().reset_index(drop=True)
if len(SSL_META)!=200000 or SSL_META.sha256.duplicated().any():
    raise RuntimeError("SSL private manifest invalid.")
SSL_META["y"]=y01(SSL_META.label)

# Physical SSL order; explicitly verify no physical label column.
ssl_parts=[]
for p in sorted(ROLE_DIR["SSL"].glob("*.parquet")):
    if "label" in pq.ParquetFile(p).schema_arrow.names:
        raise RuntimeError("SSL physical file unexpectedly contains label.")
    q=pd.read_parquet(p,columns=["sha256"])
    ssl_parts.append(q)
SSL_PHYS_META=pd.concat(ssl_parts,ignore_index=True)
SSL_PHYS_META["sha256"]=SSL_PHYS_META.sha256.astype(str).str.lower()
if set(SSL_PHYS_META.sha256)!=set(SSL_META.sha256):
    raise RuntimeError("SSL physical/private SHA mismatch.")
label_map=dict(zip(SSL_META.sha256,SSL_META.y.astype(int)))
SSL_PHYS_META["y"]=SSL_PHYS_META.sha256.map(label_map).astype(int)
SSL_META=SSL_PHYS_META.copy()

# DEV is allowed before freeze.
DEV=read_role("DEV",["sha256","url","text","label","date"]+DOM_COLS)
DEV["sha256"]=DEV.sha256.astype(str).str.lower()
DEVY=y01(DEV.label)
if len(DEV)!=20000: raise RuntimeError("DEV row mismatch.")

# Fixed representation and engineering partitions.
idx=np.arange(len(DEV))
REP_CAL,REP_EVAL=train_test_split(idx,test_size=.5,stratify=DEVY,random_state=20260821)
_,eng_half=train_test_split(idx,test_size=.5,stratify=DEVY,random_state=20260812)
ENG_TUNE,ENG_META=train_test_split(
    eng_half,test_size=.5,stratify=DEVY[eng_half],random_state=20260814
)
REP_CAL=np.sort(REP_CAL); REP_EVAL=np.sort(REP_EVAL)
ENG_TUNE=np.sort(ENG_TUNE); ENG_META=np.sort(ENG_META)

def keyed_rank(sha,seed):
    return hashlib.sha256(f"{seed}|{sha}".encode()).hexdigest()

def nested_budget_indices(rank_seed):
    out={}; rank_by={}
    for cls in [0,1]:
        q=SSL_META[SSL_META.y.eq(cls)][["sha256"]].copy()
        q["idx"]=q.index.to_numpy()
        q["rank"]=[keyed_rank(x,rank_seed) for x in q.sha256]
        rank_by[cls]=q.sort_values("rank").idx.to_numpy()
    for B in BUDGETS:
        if B==200000:
            ii=np.arange(200000,dtype=int)
        else:
            n=B//2
            ii=np.sort(np.concatenate([rank_by[0][:n],rank_by[1][:n]]))
        if len(ii)!=B: raise RuntimeError(f"Budget construction failed {B}")
        out[B]=ii
    return out

CURVE_BUDGET_INDEX={rs:nested_budget_indices(rs) for rs in CURVE_LABEL_RANK_SEEDS}

def balanced_indices(rank_seed,budget):
    rank_by={}
    for cls in [0,1]:
        q=SSL_META[SSL_META.y.eq(cls)][["sha256"]].copy()
        q["idx"]=q.index.to_numpy()
        q["rank"]=[keyed_rank(x,rank_seed) for x in q.sha256]
        rank_by[cls]=q.sort_values("rank").idx.to_numpy()
    if budget==200000:
        return np.arange(200000,dtype=int)
    n=budget//2
    return np.sort(np.concatenate([rank_by[0][:n],rank_by[1][:n]]))

FF1_SELECTIONS={
    i:balanced_indices(rs,FF1_BUDGET)
    for i,rs in enumerate(FF1_LABEL_RANK_SEEDS)
}
N10_SELECTIONS={
    i:balanced_indices(rs,N10_BUDGET)
    for i,rs in enumerate(N10_LABEL_RANK_SEEDS)
}
ENGINEERING_20K_IDX=balanced_indices(ENGINEERING_RANK_SEED,20_000)

# DEV / SSL disjointness.
if set(SSL_META.sha256) & set(DEV.sha256):
    raise RuntimeError("SSL/DEV SHA overlap.")

atomic_json(AUDIT/"PRE_FREEZE_DATA_PROTOCOL.json",{
    "status":"PASS",
    "ssl_rows":len(SSL_META),
    "dev_rows":len(DEV),
    "ssl_physical_has_label":False,
    "ssl_dev_sha_disjoint":True,
    "rep_dev_split":{"threshold":len(REP_CAL),"eval":len(REP_EVAL)},
    "engineering_split":{"tune":len(ENG_TUNE),"meta":len(ENG_META)},
    "cal_final_content_accessed":False,
})
print("PRE-FREEZE DATA PROTOCOL PASS")


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 03 — URL-DAPT: fresh MLM on all 200k unlabeled URLs
HF_TOKEN=os.environ.get("HF_TOKEN")
if not HF_TOKEN and Path("/kaggle/working").exists():
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN=None

url_kwargs=hf_kwargs(URL_BASE_SRC,"bert-base-uncased")
url_tok=AutoTokenizer.from_pretrained(URL_BASE_SRC,token=HF_TOKEN,**url_kwargs)
text_tok=AutoTokenizer.from_pretrained(TEXT_BASE_SRC,token=HF_TOKEN)

URL_DAPT_DIR=CKPT/"URL_DAPT_BERT_200K"
DAPT_DONE=URL_DAPT_DIR/"COMPLETE.json"

def read_ssl_urls():
    vals=[]
    for p in sorted(ROLE_DIR["SSL"].glob("*.parquet")):
        q=pd.read_parquet(p,columns=["url"])
        vals.extend(q.url.fillna("").astype(str).tolist())
    if len(vals)!=200000: raise RuntimeError("SSL URL count mismatch.")
    return vals

class EncodedListDataset(Dataset):
    def __init__(self,enc): self.enc=enc
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self,i): return {k:self.enc[k][i] for k in self.enc}

def train_url_dapt(batch_size):
    seed_all(MASTER_SEED)
    URL_DAPT_DIR.mkdir(parents=True,exist_ok=True)
    urls=read_ssl_urls()                 # labels are never read here
    enc=url_tok(urls,truncation=True,max_length=URL_MAX_LEN,padding=False,add_special_tokens=True)
    del urls; gc.collect()

    model=AutoModelForMaskedLM.from_pretrained(
        URL_BASE_SRC,token=HF_TOKEN,**url_kwargs
    ).to(DEVICE)
    model.train()
    ds=EncodedListDataset(enc)
    coll=DataCollatorForLanguageModeling(tokenizer=url_tok,mlm_probability=URL_MLM_PROB)
    dl=DataLoader(
        ds,batch_size=batch_size,shuffle=True,
        generator=torch.Generator().manual_seed(MASTER_SEED),
        collate_fn=coll,num_workers=0
    )
    opt=torch.optim.AdamW(model.parameters(),lr=URL_DAPT_LR,weight_decay=URL_DAPT_WEIGHT_DECAY)
    total=len(dl)*URL_DAPT_EPOCHS
    sched=get_linear_schedule_with_warmup(opt,int(URL_DAPT_WARMUP_FRAC*total),total)
    scaler=torch.cuda.amp.GradScaler(enabled=AMP)
    losses=[]; t0=time.time()

    for ep in range(URL_DAPT_EPOCHS):
        for step,b in enumerate(dl,1):
            b={k:v.to(DEVICE) for k,v in b.items()}
            opt.zero_grad(set_to_none=True)
            with amp_ctx(): loss=model(**b).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sched.step()
            losses.append(float(loss.detach().cpu()))
            if step%500==0 or step==len(dl):
                print({
                    "URL_DAPT_step":step,"of":len(dl),
                    "loss100":round(float(np.mean(losses[-100:])),4),
                    "elapsed_min":round((time.time()-t0)/60,2)
                })

    # Save adapted encoder only.
    model.base_model.save_pretrained(URL_DAPT_DIR)
    url_tok.save_pretrained(URL_DAPT_DIR)
    info={
        "status":"COMPLETE","fresh_from_base":True,"source":str(URL_BASE_SRC),
        "rows":URL_DAPT_ROWS,"epochs":URL_DAPT_EPOCHS,"batch":batch_size,
        "lr":URL_DAPT_LR,"mlm_probability":URL_MLM_PROB,
        "mean_train_loss":float(np.mean(losses)),
        "last100_train_loss":float(np.mean(losses[-100:])),
        "wall_minutes":float((time.time()-t0)/60),
        "uses_labels":False,
    }
    atomic_json(DAPT_DONE,info)
    del model,ds,dl,enc
    gc.collect(); torch.cuda.empty_cache()
    return info

if DAPT_DONE.exists():
    DAPT_INFO=json.loads(DAPT_DONE.read_text())
    print("Reuse completed URL-DAPT checkpoint.")
else:
    last=None
    for bs in URL_DAPT_BATCH_CANDIDATES:
        try:
            DAPT_INFO=train_url_dapt(bs); last=None; break
        except torch.cuda.OutOfMemoryError as e:
            last=e; gc.collect(); torch.cuda.empty_cache()
            print({"URL_DAPT_OOM":bs,"retry":"smaller"})
    if last is not None: raise last

URL_DAPT_SRC=str(URL_DAPT_DIR)
url_dapt_tok=AutoTokenizer.from_pretrained(URL_DAPT_SRC)
REP_SRC={
    "R0":(URL_BASE_SRC,TEXT_BASE_SRC),
    "RU":(URL_DAPT_SRC,TEXT_BASE_SRC),
    "RT":(URL_BASE_SRC,TEXT_DAPT_SRC),
    "RUT":(URL_DAPT_SRC,TEXT_DAPT_SRC),
}
atomic_json(AUDIT/"REPRESENTATION_SOURCES.json",{k:list(v) for k,v in REP_SRC.items()})
print(json.dumps(DAPT_INFO,indent=2))


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 04 — Gemeinsame Metriken und SSL/DEV-Embedding-Cache
def masked_mean(h,m):
    mm=m.unsqueeze(-1).to(h.dtype)
    return (h*mm).sum(1)/mm.sum(1).clamp_min(1)
def pooled(out,mask):
    return masked_mean(out.last_hidden_state,mask)

def thr_fpr(neg_scores,f):
    s=np.asarray(neg_scores,dtype=np.float64)
    if not np.isfinite(s).all(): raise RuntimeError("non-finite threshold scores")
    k=int(math.floor(float(f)*len(s)+1e-12))
    if k<=0: return float(np.nextafter(s.max(),np.inf))
    ss=np.sort(s)
    return float(np.nextafter(ss[-k],np.inf))

def ci_fpr(fp,n):
    if n==0:return (np.nan,np.nan)
    return (
        0. if fp==0 else float(beta_dist.ppf(.025,fp,n-fp+1)),
        1. if fp==n else float(beta_dist.ppf(.975,fp+1,n-fp))
    )

def op(y,score,th):
    y=np.asarray(y,int); s=np.asarray(score,float); p=s>=th
    tp=int(((p)&(y==1)).sum()); fp=int(((p)&(y==0)).sum())
    tn=int(((~p)&(y==0)).sum()); fn=int(((~p)&(y==1)).sum())
    tpr=tp/max(tp+fn,1); fpr=fp/max(fp+tn,1); prec=tp/max(tp+fp,1)
    lo,hi=ci_fpr(fp,fp+tn)
    return {
        "tpr":tpr,"fpr":fpr,"precision":prec,
        "f1":2*prec*tpr/max(prec+tpr,1e-12),
        "tp":tp,"fp":fp,"tn":tn,"fn":fn,
        "fp_per_1000":1000*fpr,"fpr_ci_lo":lo,"fpr_ci_hi":hi,
    }

def precision_at_recall(y,s,target=.90):
    p,r,_=precision_recall_curve(y,s)
    ok=np.where(r>=target)[0]
    return float(np.max(p[ok])) if len(ok) else np.nan

def curves(y,s):
    y=np.asarray(y,int); s=np.asarray(s,float)
    fpr,tpr,_=roc_curve(y,s)
    def fat(target):
        ii=np.where(tpr>=target)[0]
        return float(fpr[ii[0]]) if len(ii) else np.nan
    return {
        "AP":float(average_precision_score(y,s)),
        "AUC":float(roc_auc_score(y,s)),
        "P_at_R90":precision_at_recall(y,s,.90),
        "FPR_at_TPR90":fat(.90),
        "FPR_at_TPR95":fat(.95),
    }

def exact_signflip_p(diff):
    d=np.asarray(diff,float); d=d[np.isfinite(d)]
    if len(d)==0:return np.nan
    obs=abs(d.mean())
    vals=[
        abs(np.mean(d*np.asarray(signs)))
        for signs in itertools.product([-1,1],repeat=len(d))
    ]
    return float(np.mean(np.asarray(vals)>=obs-1e-15))

def mean_ci(diff):
    d=np.asarray(diff,float); d=d[np.isfinite(d)]
    if len(d)<2:
        return (float(d.mean()) if len(d) else np.nan,np.nan,np.nan)
    m=float(d.mean()); se=float(d.std(ddof=1)/math.sqrt(len(d)))
    q=float(student_t.ppf(.975,len(d)-1))
    return m,m-q*se,m+q*se

def holm_adjust(p_values):
    p=np.asarray(p_values,float)
    out=np.full(len(p),np.nan)
    finite=np.flatnonzero(np.isfinite(p))
    vals=p[finite]
    order=np.argsort(vals,kind="mergesort")
    running=0.
    for rank,pos in enumerate(order):
        running=max(running,(len(vals)-rank)*vals[pos])
        out[finite[pos]]=min(1.,running)
    # monotonicity in sorted order
    sorted_adj=[out[finite[pos]] for pos in order]
    for i in range(1,len(sorted_adj)):
        sorted_adj[i]=max(sorted_adj[i],sorted_adj[i-1])
    for pos,val in zip(order,sorted_adj):
        out[finite[pos]]=val
    return out

# Embedding cache is pre-freeze restricted to SSL and DEV.
STREAMS={
    "URL_BASE":(URL_BASE_SRC,url_tok,"url",URL_MAX_LEN,EMBED_BATCH_URL),
    "URL_DAPT":(URL_DAPT_SRC,url_dapt_tok,"url",URL_MAX_LEN,EMBED_BATCH_URL),
    "TEXT_BASE":(TEXT_BASE_SRC,text_tok,"text",TEXT_MAX_LEN,EMBED_BATCH_TEXT),
    "TEXT_DAPT":(TEXT_DAPT_SRC,text_tok,"text",TEXT_MAX_LEN,EMBED_BATCH_TEXT),
}

@torch.no_grad()
def embed_stream_role(stream,role):
    if role not in {"SSL","DEV","CAL","FINAL"}: raise KeyError(role)
    if role in {"CAL","FINAL"}: _guard_role(role)
    src,tok,col,max_len,batch=STREAMS[stream]
    out=EMB/f"{stream}_{role}.npy"
    done=EMB/f"{stream}_{role}.complete.json"
    prog=EMB/f"{stream}_{role}.progress.json"
    if done.exists() and out.exists(): return out

    if role=="SSL":
        vals=[]; shas=[]
        for p in sorted(ROLE_DIR["SSL"].glob("*.parquet")):
            q=pd.read_parquet(p,columns=["sha256",col])
            shas.extend(q.sha256.astype(str).str.lower().tolist())
            vals.extend(q[col].fillna("").astype(str).tolist())
        if len(vals)!=200000: raise RuntimeError("SSL embed role mismatch")
    else:
        q=read_role(role,["sha256",col])
        shas=q.sha256.astype(str).str.lower().tolist()
        vals=q[col].fillna("").astype(str).tolist()

    kwargs=hf_kwargs(src,"bert-base-uncased") if stream=="URL_BASE" else {}
    model=AutoModel.from_pretrained(src,token=HF_TOKEN,**kwargs).to(DEVICE).eval()
    hdim=int(model.config.hidden_size)
    n=len(vals); start=0
    if prog.exists() and out.exists():
        try:start=int(json.loads(prog.read_text()).get("next_row",0))
        except:start=0
    if start==0:
        out.unlink(missing_ok=True)
        arr=np.lib.format.open_memmap(out,mode="w+",dtype=np.float32,shape=(n,hdim))
    else:
        arr=np.load(out,mmap_mode="r+")
    for st in range(start,n,batch):
        en=min(st+batch,n)
        b=tok(vals[st:en],padding=True,truncation=True,max_length=max_len,return_tensors="pt")
        b={k:v.to(DEVICE) for k,v in b.items()}
        with amp_ctx():z=pooled(model(**b),b["attention_mask"])
        arr[st:en]=z.float().cpu().numpy()
        if en%5000<batch or en==n:
            arr.flush(); atomic_json(prog,{"next_row":en})
            print({"EMBED":stream,"role":role,"rows":en,"total":n})
    arr.flush()
    atomic_json(done,{"status":"COMPLETE","rows":n,"hidden_size":hdim,"source":str(src)})
    prog.unlink(missing_ok=True)
    del arr,model,vals,shas
    gc.collect();torch.cuda.empty_cache()
    return out

for stream in STREAMS:
    for role in ["SSL","DEV"]:
        embed_stream_role(stream,role)

def emb_mm(stream,role):
    return np.load(EMB/f"{stream}_{role}.npy",mmap_mode="r")

def streams_for_rep(rep):
    if rep=="R0": return "URL_BASE","TEXT_BASE"
    if rep=="RU": return "URL_DAPT","TEXT_BASE"
    if rep=="RT": return "URL_BASE","TEXT_DAPT"
    if rep=="RUT": return "URL_DAPT","TEXT_DAPT"
    raise KeyError(rep)

def take_features(rep,role,idx=None):
    us,ts=streams_for_rep(rep)
    u=emb_mm(us,role); t=emb_mm(ts,role)
    if idx is None: idx=np.arange(len(u))
    return np.concatenate([
        np.asarray(u[idx],np.float32),
        np.asarray(t[idx],np.float32)
    ],axis=1)

FEATURE_DIM=take_features("R0","DEV",np.arange(1)).shape[1]
print({"PRE_FREEZE_EMBEDDINGS":"COMPLETE","feature_dim":FEATURE_DIM})


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 05 — FF1: 2×2 URL×HTML DAPT-Ablation auf DEV, N5, Linear + MLP
class MLPProbe(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(d,256),nn.GELU(),nn.Dropout(.15),nn.Linear(256,1)
        )
    def forward(self,x):return self.net(x).squeeze(-1)

def fit_mlp(X,y,seed):
    seed_all(seed)
    m=MLPProbe(X.shape[1]).to(DEVICE)
    opt=torch.optim.AdamW(m.parameters(),lr=1e-3,weight_decay=1e-3)
    xx=torch.tensor(np.asarray(X,np.float32))
    yy=torch.tensor(np.asarray(y),dtype=torch.float32)
    dl=DataLoader(
        TensorDataset(xx,yy),batch_size=MLP_BATCH,shuffle=True,
        generator=torch.Generator().manual_seed(seed),num_workers=0
    )
    for _ in range(MLP_EPOCHS):
        m.train()
        for a,b in dl:
            a=a.to(DEVICE);b=b.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=F.binary_cross_entropy_with_logits(m(a),b)
            loss.backward();opt.step()
    return m.eval()

@torch.no_grad()
def score_mlp_features(m,X,chunk=4096):
    out=np.empty(len(X),np.float32)
    for st in range(0,len(X),chunk):
        en=min(st+chunk,len(X))
        out[st:en]=m(torch.tensor(np.asarray(X[st:en],np.float32),device=DEVICE)).float().cpu().numpy()
    return out

FF1_PARTS=RESULTS/"ff1_parts";FF1_PARTS.mkdir(exist_ok=True)
for run_i,(mseed,rseed) in enumerate(zip(FF1_MODEL_SEEDS,FF1_LABEL_RANK_SEEDS)):
    train_idx=FF1_SELECTIONS[run_i]
    ytr=SSL_META.y.iloc[train_idx].to_numpy(int)
    for rep in ["R0","RU","RT","RUT"]:
        Xtr=take_features(rep,"SSL",train_idx)
        Xdev=take_features(rep,"DEV")
        for probe in ["LINEAR","MLP"]:
            part=FF1_PARTS/f"run{run_i}_{rep}_{probe}.csv"
            if part.exists():continue
            if probe=="LINEAR":
                clf=LogisticRegression(
                    C=1,max_iter=2000,solver="liblinear",
                    class_weight="balanced",random_state=mseed
                ).fit(Xtr,ytr)
                s=clf.decision_function(Xdev)
            else:
                m=fit_mlp(Xtr,ytr,mseed)
                s=score_mlp_features(m,Xdev)
                del m;torch.cuda.empty_cache()
            yc=DEVY[REP_CAL];ye=DEVY[REP_EVAL]
            th=thr_fpr(s[REP_CAL][yc==0],PRIMARY_FPR)
            row={
                "run":run_i,"model_seed":mseed,"label_rank_seed":rseed,
                "rep":rep,"probe":probe,"budget":FF1_BUDGET,
                "dataset":"DEV_EVAL","scenario":"DEV_EVAL",
                "target_fpr":PRIMARY_FPR,"threshold":th,
                **op(ye,s[REP_EVAL],th),**curves(ye,s[REP_EVAL])
            }
            pd.DataFrame([row]).to_csv(part,index=False)
        del Xtr,Xdev;gc.collect()

FF1_ALL=pd.concat([pd.read_csv(p) for p in sorted(FF1_PARTS.glob("*.csv"))],ignore_index=True)
FF1_ALL.to_csv(RESULTS/"FF1_MODALITY_N5_ALL.csv",index=False)

FF1_AGG=FF1_ALL.groupby(["rep","probe"],as_index=False).agg(
    n=("tpr","size"),tpr_mean=("tpr","mean"),tpr_sd=("tpr","std"),
    fpr_mean=("fpr","mean"),precision_mean=("precision","mean"),
    AP_mean=("AP","mean"),P_at_R90_mean=("P_at_R90","mean"),
    FPR_at_TPR90_mean=("FPR_at_TPR90","mean"),
)
FF1_AGG.to_csv(TABLES/"TABLE_FF1_MODALITY_DAPT_DEV_N5.csv",index=False)

contrast_specs=[
    ("RU-R0","RU","R0"),
    ("RT-R0","RT","R0"),
    ("RUT-R0","RUT","R0"),
    ("RUT-RT","RUT","RT"),
    ("RUT-RU","RUT","RU"),
]
cres=[]
for probe in ["LINEAR","MLP"]:
    g=FF1_ALL[FF1_ALL.probe.eq(probe)]
    for name,a,b in contrast_specs:
        piv=g[g.rep.isin([a,b])].pivot(index="run",columns="rep",values="tpr")
        d=(piv[a]-piv[b]).to_numpy(float)
        m,lo,hi=mean_ci(d)
        cres.append({
            "probe":probe,"contrast":name,"n":len(d),
            "mean_delta_tpr_pp":100*m,
            "ci95_lo_pp":100*lo,"ci95_hi_pp":100*hi,
            "positive_pairs":int((d>0).sum()),
            "negative_pairs":int((d<0).sum()),
            "exact_signflip_p":exact_signflip_p(d),
        })
        if name=="RUT-RT":
            d_inc=d.copy()
        if name=="RU-R0":
            d_iso=d.copy()
    # Factorial URL×HTML interaction: URL effect after HTML minus URL isolated effect.
    d=d_inc-d_iso
    m,lo,hi=mean_ci(d)
    cres.append({
        "probe":probe,"contrast":"INTERACTION_URLxHTML","n":len(d),
        "mean_delta_tpr_pp":100*m,
        "ci95_lo_pp":100*lo,"ci95_hi_pp":100*hi,
        "positive_pairs":int((d>0).sum()),
        "negative_pairs":int((d<0).sum()),
        "exact_signflip_p":exact_signflip_p(d),
    })
FF1_STATS=pd.DataFrame(cres)
FF1_STATS.to_csv(TABLES/"TABLE_FF1_MODALITY_PAIRED_STATS.csv",index=False)

# Historical contrastive evidence is retained separately, not mixed into this new DEV experiment.
HIST_FF1_PRIMARY=pd.read_csv(HIST/"HIST_FF1_PRIMARY.csv")
HIST_FF1_STATS=pd.read_csv(HIST/"HIST_FF1_PAIRED_STATS.csv")
HIST_FF1_PRIMARY.to_csv(TABLES/"TABLE_FF1_CONTRASTIVE_HISTORICAL_PRIMARY.csv",index=False)
HIST_FF1_STATS.to_csv(TABLES/"TABLE_FF1_CONTRASTIVE_HISTORICAL_PAIRED.csv",index=False)

print("=== FF1 modality means ===")
display(FF1_AGG)
print("=== FF1 modality paired contrasts ===")
display(FF1_STATS)


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 06 — FF2: Label-Effizienz R0/RU/RT/RUT, N5; zunächst DEV-only
CURVE_PARTS=RESULTS/"curve_parts";CURVE_PARTS.mkdir(exist_ok=True)
PROBE_CKPT=CKPT/"CURVE"

def score_linear_features(clf,X):
    return clf.decision_function(X).astype(np.float32)

for run_i,(mseed,rseed) in enumerate(zip(CURVE_MODEL_SEEDS,CURVE_LABEL_RANK_SEEDS)):
    for B in BUDGETS:
        train_idx=CURVE_BUDGET_INDEX[rseed][B]
        ytr=SSL_META.y.iloc[train_idx].to_numpy(int)
        for rep in ["R0","RU","RT","RUT"]:
            Xtr=take_features(rep,"SSL",train_idx)
            Xdev=take_features(rep,"DEV")
            for probe in ["LINEAR","MLP"]:
                part=CURVE_PARTS/f"dev_run{run_i}_{rep}_{probe}_{B}.csv"
                cdir=PROBE_CKPT/f"run{run_i}"/rep/f"B{B}"
                cdir.mkdir(parents=True,exist_ok=True)
                if part.exists():continue
                if probe=="LINEAR":
                    mp=cdir/"linear.npz"
                    if mp.exists():
                        z=np.load(mp)
                        coef=z["coef"];intercept=z["intercept"]
                        s=(Xdev@coef.T).reshape(-1)+float(intercept.reshape(-1)[0])
                    else:
                        clf=LogisticRegression(
                            C=1,max_iter=2000,solver="liblinear",
                            class_weight="balanced",random_state=mseed
                        ).fit(Xtr,ytr)
                        np.savez(mp,coef=clf.coef_.astype(np.float32),intercept=clf.intercept_.astype(np.float32))
                        s=clf.decision_function(Xdev)
                else:
                    mp=cdir/"mlp.pt"
                    if mp.exists():
                        m=MLPProbe(FEATURE_DIM).to(DEVICE)
                        m.load_state_dict(torch.load(mp,map_location=DEVICE));m.eval()
                    else:
                        m=fit_mlp(Xtr,ytr,mseed);atomic_torch(mp,m.state_dict())
                    s=score_mlp_features(m,Xdev);del m;torch.cuda.empty_cache()

                yc=DEVY[REP_CAL];ye=DEVY[REP_EVAL]
                rows=[]
                for f in TARGET_FPRS:
                    th=thr_fpr(s[REP_CAL][yc==0],f)
                    rows.append({
                        "run":run_i,"model_seed":mseed,"label_rank_seed":rseed,
                        "budget":B,"rep":rep,"probe":probe,
                        "dataset":"DEV_EVAL","scenario":"DEV_EVAL",
                        "target_fpr":f,"threshold":th,
                        **op(ye,s[REP_EVAL],th),**curves(ye,s[REP_EVAL])
                    })
                pd.DataFrame(rows).to_csv(part,index=False)
            del Xtr,Xdev;gc.collect()

DEV_CURVE=pd.concat([pd.read_csv(p) for p in sorted(CURVE_PARTS.glob("dev_*.csv"))],ignore_index=True)
DEV_CURVE.to_csv(RESULTS/"FF2_LABEL_CURVE_DEV_ALL.csv",index=False)

DEV_PRIMARY=DEV_CURVE[np.isclose(DEV_CURVE.target_fpr,PRIMARY_FPR)]
DEV_AGG=DEV_PRIMARY.groupby(["rep","probe","budget"],as_index=False).agg(
    tpr_mean=("tpr","mean"),tpr_sd=("tpr","std"),
    fpr_mean=("fpr","mean"),AP_mean=("AP","mean"),P_at_R90_mean=("P_at_R90","mean")
)
DEV_AGG.to_csv(TABLES/"TABLE_FF2_LABEL_EFFICIENCY_DEV.csv",index=False)

# B95 relative to own 200k DEV reference.
b95_rows=[]
for (rep,probe),g in DEV_AGG.groupby(["rep","probe"]):
    g=g.sort_values("budget")
    ref=float(g.loc[g.budget.eq(200000),"tpr_mean"].iloc[0])
    target=RETENTION_TARGET*ref
    ok=g[g.tpr_mean>=target]
    b95=int(ok.budget.min()) if len(ok) else np.nan
    b95_rows.append({
        "rep":rep,"probe":probe,"reference_tpr_dev":ref,
        "retention_target":RETENTION_TARGET,"target_tpr":target,"B95_dev":b95
    })
B95=pd.DataFrame(b95_rows)
B95.to_csv(TABLES/"TABLE_FF2_B95_DEV.csv",index=False)
display(B95)


# Marginal label utility: same analytical layer as historical v8, now for R0/RU/RT/RUT.
marginal=[]
for (rep,probe),g in DEV_AGG.groupby(["rep","probe"]):
    g=g.sort_values("budget").reset_index(drop=True)
    for i in range(1,len(g)):
        b0=int(g.loc[i-1,"budget"]); b1=int(g.loc[i,"budget"])
        dpp=100*float(g.loc[i,"tpr_mean"]-g.loc[i-1,"tpr_mean"])
        marginal.append({
            "rep":rep,"probe":probe,
            "from_budget":b0,"to_budget":b1,
            "delta_tpr_pp":dpp,
            "pp_per_10k_labels":dpp/(b1-b0)*10000,
        })
MARGINAL=pd.DataFrame(marginal)
MARGINAL.to_csv(TABLES/"TABLE_FF2_MARGINAL_LABEL_UTILITY_DEV.csv",index=False)


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 07 — Deep DUAL/TRI, DOM-GCN und allgemeine Trainings-/Scoring-Funktionen
def freeze_last(model,n):
    for p in model.parameters():p.requires_grad=False
    layers=getattr(getattr(model,"encoder",None),"layer",None)
    if layers is None:
        layers=getattr(getattr(model,"transformer",None),"layer",None)
    if layers is None: raise RuntimeError(type(model))
    for layer in layers[-n:]:
        for p in layer.parameters():p.requires_grad=True
    if getattr(model,"pooler",None) is not None:
        for p in model.pooler.parameters():p.requires_grad=True

# Check hidden dimensions.
_u=AutoModel.from_pretrained(URL_BASE_SRC,token=HF_TOKEN,**hf_kwargs(URL_BASE_SRC,"bert-base-uncased"))
_t=AutoModel.from_pretrained(TEXT_BASE_SRC,token=HF_TOKEN)
URL_H=int(_u.config.hidden_size);TEXT_H=int(_t.config.hidden_size)
del _u,_t;gc.collect()

DOM_VOCAB=json.loads(DOM_VOCAB_PATH.read_text())
PAD,UNK,MASK=0,1,2

def bucket(v):
    try:return min(max(int(v),0),31)
    except:return 0

def _rv(r,k):
    if isinstance(r,dict):return r.get(k)
    try:return r[k]
    except:return getattr(r,k,None)

def graph_row(r):
    def arr(k):
        x=_rv(r,k)
        if isinstance(x,np.ndarray):return x.tolist()
        if isinstance(x,list):return x
        try:return list(x)
        except:return []
    tags=arr("dom_tag");par=arr("dom_parent_idx");dep=arr("dom_depth")
    att=arr("dom_attr_count");chi=arr("dom_child_count")
    n=min(len(tags),DOM_MAX_NODES)
    if n==0:
        return (
            np.array([PAD],np.int64),np.array([0],np.int64),
            np.array([0],np.int64),np.array([0],np.int64),
            np.zeros((0,2),np.int64)
        )
    tag=np.array([DOM_VOCAB.get(str(x),UNK) for x in tags[:n]],np.int64)
    depth=np.array([bucket(x) for x in dep[:n]],np.int64)
    attr=np.array([bucket(x) for x in att[:n]],np.int64)
    child=np.array([bucket(x) for x in chi[:n]],np.int64)
    edges=[]
    for ii,pp in enumerate(par[:n]):
        try:pp=int(pp)
        except:continue
        if ii>0 and 0<=pp<n:edges.extend([(pp,ii),(ii,pp)])
    return tag,depth,attr,child,np.asarray(edges,np.int64)

def graph_batch(rows):
    T=[];D=[];A=[];C=[];E=[];B=[];off=0
    row_iter=rows.itertuples(index=False) if isinstance(rows,pd.DataFrame) else rows
    n_graphs=len(rows)
    for bi,r in enumerate(row_iter):
        t,d,a,c,e=graph_row(r);n=len(t)
        T.append(t);D.append(d);A.append(a);C.append(c);B.append(np.full(n,bi,np.int64))
        if len(e):E.append(e+off)
        off+=n
    def cat(xs):
        return torch.tensor(np.concatenate(xs),dtype=torch.long,device=DEVICE)
    edge=torch.tensor(
        np.concatenate(E,0).T if E else np.zeros((2,0),np.int64),
        dtype=torch.long,device=DEVICE
    )
    return {
        "tag":cat(T),"depth":cat(D),"attr":cat(A),"child":cat(C),
        "edge":edge,"batch":cat(B),"n_graphs":n_graphs
    }

class GCNLayer(nn.Module):
    def __init__(self,d):
        super().__init__();self.s=nn.Linear(d,d);self.n=nn.Linear(d,d);self.norm=nn.LayerNorm(d)
    def forward(self,x,e):
        if e.numel()==0:return self.norm(x+F.gelu(self.s(x)))
        src,dst=e;agg=torch.zeros_like(x)
        deg=torch.zeros((len(x),1),device=x.device,dtype=x.dtype)
        agg.index_add_(0,dst,x[src])
        deg.index_add_(0,dst,torch.ones((len(dst),1),device=x.device,dtype=x.dtype))
        return self.norm(x+F.gelu(self.s(x)+self.n(agg/deg.clamp_min(1))))

class DOMEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tag=nn.Embedding(len(DOM_VOCAB),96,padding_idx=PAD)
        self.depth=nn.Embedding(32,16);self.attr=nn.Embedding(32,8);self.child=nn.Embedding(32,8)
        self.inp=nn.Linear(128,DOM_DIM)
        self.layers=nn.ModuleList([GCNLayer(DOM_DIM) for _ in range(DOM_LAYERS)])
        self.att=nn.Linear(DOM_DIM,1)
    def nodes(self,g):
        x=self.inp(torch.cat([
            self.tag(g["tag"]),self.depth(g["depth"]),
            self.attr(g["attr"]),self.child(g["child"])
        ],1))
        for l in self.layers:x=l(x,g["edge"])
        return x
    def forward(self,g):
        x=self.nodes(g);n=g["n_graphs"]
        num=torch.zeros((n,DOM_DIM),device=x.device)
        den=torch.zeros((n,1),device=x.device)
        w=torch.exp(torch.clamp(self.att(x).squeeze(-1),-10,10))
        num.index_add_(0,g["batch"],x*w[:,None])
        den.index_add_(0,g["batch"],w[:,None])
        return num/den.clamp_min(1e-8)

# Hard checkpoint compatibility preflight.
_dom=DOMEncoder()
_dom.load_state_dict(torch.load(DOM_ENCODER_PATH,map_location="cpu"),strict=True)
del _dom

class DeepFusion(nn.Module):
    def __init__(self,rep,use_dom):
        super().__init__();self.rep=rep;self.use_dom=use_dom
        us,ts=REP_SRC[rep]
        ukw=hf_kwargs(us,"bert-base-uncased") if rep in {"R0","RT"} else {}
        self.u=AutoModel.from_pretrained(us,token=HF_TOKEN,**ukw)
        self.t=AutoModel.from_pretrained(ts,token=HF_TOKEN)
        freeze_last(self.u,DEEP_LAST_N);freeze_last(self.t,DEEP_LAST_N)
        self.ua=nn.Linear(int(self.u.config.hidden_size),PROJ_DIM)
        self.ta=nn.Linear(int(self.t.config.hidden_size),PROJ_DIM)
        self.dom=DOMEncoder() if use_dom else None
        if use_dom:
            self.dom.load_state_dict(torch.load(DOM_ENCODER_PATH,map_location="cpu"))
        self.da=nn.Linear(DOM_DIM,PROJ_DIM) if use_dom else None
        self.gate=nn.Sequential(nn.Linear(PROJ_DIM,64),nn.GELU(),nn.Linear(64,1))
        nm=3 if use_dom else 2
        self.head=nn.Sequential(
            nn.Linear(PROJ_DIM*(nm+1),512),nn.GELU(),nn.Dropout(.2),
            nn.Linear(512,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1)
        )
        self.uaux=nn.Linear(PROJ_DIM,1)
        self.taux=nn.Linear(PROJ_DIM,1)
        self.daux=nn.Linear(PROJ_DIM,1) if use_dom else None

    def encode(self,u,t,g=None):
        zu=F.normalize(self.ua(pooled(self.u(**u),u["attention_mask"])),dim=-1)
        zt=F.normalize(self.ta(pooled(self.t(**t),t["attention_mask"])),dim=-1)
        zs=[zu,zt]
        if self.use_dom:zs.append(F.normalize(self.da(self.dom(g)),dim=-1))
        return zs

    def forward(self,u,t,g=None,fusion_mode="gated"):
        zs=self.encode(u,t,g);st=torch.stack(zs,1)
        gw=torch.softmax(self.gate(st).squeeze(-1),1)
        weighted=(st*gw[:,:,None]).sum(1) if fusion_mode=="gated" else st.mean(1)
        main=self.head(torch.cat(zs+[weighted],1)).squeeze(-1)
        aux={"url":self.uaux(zs[0]).squeeze(-1),"text":self.taux(zs[1]).squeeze(-1)}
        if self.use_dom:aux["dom"]=self.daux(zs[2]).squeeze(-1)
        return main,aux,gw

def _wb(logits,y):
    return F.binary_cross_entropy_with_logits(logits,y)

def deep_loss(main,aux,y):
    return _wb(main,y)+AUX_TOTAL_WEIGHT*torch.stack([_wb(a,y) for a in aux.values()]).mean()

def tok_url_values(values,tokenizer):
    b=tokenizer(values,padding=True,truncation=True,max_length=URL_MAX_LEN,return_tensors="pt")
    return {k:v.to(DEVICE,non_blocking=True) for k,v in b.items()}
def tok_text_values(values):
    b=text_tok(values,padding=True,truncation=True,max_length=TEXT_MAX_LEN,return_tensors="pt")
    return {k:v.to(DEVICE,non_blocking=True) for k,v in b.items()}

# Physical SSL row lookup is resolved via SHA filtering for 20k and streaming all rows for 200k.
class DeepTrainIter(IterableDataset):
    def __init__(self,allowed_sha,seed,use_dom):
        self.allowed_sha=allowed_sha
        self.seed=seed;self.use_dom=use_dom
        self.files=sorted(ROLE_DIR["SSL"].glob("*.parquet"))
    def __iter__(self):
        rng=np.random.default_rng(self.seed)
        files=list(self.files);rng.shuffle(files)
        for p in files:
            cols=["sha256","url","text"]+(DOM_COLS if self.use_dom else [])
            q=pd.read_parquet(p,columns=cols)
            q["sha256"]=q.sha256.astype(str).str.lower()
            if self.allowed_sha is not None:
                q=q[q.sha256.isin(self.allowed_sha)]
            ii=np.arange(len(q));rng.shuffle(ii)
            for j in ii:
                r=q.iloc[j];sha=str(r.sha256)
                y=label_map.get(sha)
                if y is None:raise RuntimeError(f"Missing SSL label: {sha}")
                yield r,int(y)

def deep_collate(rows,use_dom,rep):
    rs=[x[0] for x in rows]
    y=torch.tensor([x[1] for x in rows],dtype=torch.float32,device=DEVICE)
    url_tokenizer=url_dapt_tok if rep in {"RU","RUT"} else url_tok
    u=tok_url_values([str(r.url) if pd.notna(r.url) else "" for r in rs],url_tokenizer)
    t=tok_text_values([str(r.text) if pd.notna(r.text) else "" for r in rs])
    g=graph_batch(rs) if use_dom else None
    return u,t,g,y

def train_deep(tag,rep,budget,use_dom,seed,allowed_idx=None,keep_state=True):
    out=CKPT/"DEEP"/tag
    done=out/"COMPLETE.json";state=out/"model_state.pt"
    if done.exists() and state.exists():
        return out
    out.mkdir(parents=True,exist_ok=True)
    allowed_sha=None if budget==200000 else set(SSL_META.sha256.iloc[np.asarray(allowed_idx,int)])
    cands=TRI_BATCH_CANDIDATES if use_dom else DUAL_BATCH_CANDIDATES
    last=None
    for bs in cands:
        try:
            seed_all(seed)
            m=DeepFusion(rep,use_dom).to(DEVICE)
            enc=[];head=[]
            for n,p in m.named_parameters():
                if not p.requires_grad:continue
                (enc if n.startswith("u.") or n.startswith("t.") else head).append(p)
            opt=torch.optim.AdamW([
                {"params":enc,"lr":ENCODER_LR,"weight_decay":WEIGHT_DECAY},
                {"params":head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
            ])
            steps=math.ceil(budget/bs)*DEEP_EPOCHS
            sched=get_linear_schedule_with_warmup(opt,int(WARMUP_RATIO*steps),steps)
            scaler=torch.cuda.amp.GradScaler(enabled=AMP)
            ds=DeepTrainIter(allowed_sha,seed,use_dom)
            dl=DataLoader(
                ds,batch_size=bs,
                collate_fn=lambda x:deep_collate(x,use_dom,rep),
                num_workers=0
            )
            losses=[];seen=0;t0=time.time()
            m.train()
            for u,t,g,y in dl:
                if seen>=budget:break
                opt.zero_grad(set_to_none=True)
                with amp_ctx():
                    main,aux,_=m(u,t,g)
                    loss=deep_loss(main,aux,y)
                if not torch.isfinite(loss):raise RuntimeError(f"nonfinite loss {tag}")
                scaler.scale(loss).backward();scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(),1.)
                scaler.step(opt);scaler.update();sched.step()
                seen+=len(y);losses.append(float(loss.detach().cpu()))
                if len(losses)%250==0:
                    print({"TRAIN":tag,"seen":seen,"of":budget,"loss100":float(np.mean(losses[-100:]))})
                if seen>=budget:break
            atomic_torch(state,m.state_dict())
            atomic_json(done,{
                "status":"COMPLETE","tag":tag,"rep":rep,"budget":budget,
                "use_dom":use_dom,"seed":seed,"batch_size":bs,
                "mean_loss":float(np.mean(losses)),
                "elapsed_min":float((time.time()-t0)/60),
            })
            del m,ds,dl,opt,sched
            gc.collect();torch.cuda.empty_cache()
            last=None;break
        except torch.cuda.OutOfMemoryError as e:
            last=e;gc.collect();torch.cuda.empty_cache()
            print({"DEEP_OOM":tag,"batch":bs})
    if last is not None:raise last
    return out

def load_deep(d,rep,use_dom):
    m=DeepFusion(rep,use_dom).to(DEVICE)
    m.load_state_dict(torch.load(Path(d)/"model_state.pt",map_location=DEVICE))
    m.eval();return m

@torch.no_grad()
def deep_eval_rows(model,df,use_dom,rep,batch,return_equal=True):
    out={k:np.empty(len(df),np.float32) for k in ["gated","url","text"]+(
        ["equal"] if return_equal else []
    )+(
        ["dom"] if use_dom else []
    )}
    url_tokenizer=url_dapt_tok if rep in {"RU","RUT"} else url_tok
    for st in range(0,len(df),batch):
        en=min(st+batch,len(df));rows=df.iloc[st:en]
        u=tok_url_values(rows.url.fillna("").astype(str).tolist(),url_tokenizer)
        t=tok_text_values(rows.text.fillna("").astype(str).tolist())
        g=graph_batch(rows) if use_dom else None
        with amp_ctx():
            main,aux,_=model(u,t,g,"gated")
            out["gated"][st:en]=main.float().cpu().numpy()
            out["url"][st:en]=aux["url"].float().cpu().numpy()
            out["text"][st:en]=aux["text"].float().cpu().numpy()
            if use_dom:out["dom"][st:en]=aux["dom"].float().cpu().numpy()
            if return_equal:
                eq,_,_=model(u,t,g,"equal")
                out["equal"][st:en]=eq.float().cpu().numpy()
    return out

def score_deep_adaptive(model,df,use_dom,rep,return_equal=True):
    last=None
    for bs in SCORE_BATCH_CANDIDATES:
        try:
            return deep_eval_rows(model,df,use_dom,rep,bs,return_equal),bs
        except torch.cuda.OutOfMemoryError as e:
            last=e;gc.collect();torch.cuda.empty_cache()
    raise last


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 08 — FF2 Deep-N3-RUT sowie FF4 DEV-only Architecture Freeze
# Historical Deep-N3 R0/RT are immutable embedded references.
HIST_N3=pd.read_csv(HIST/"HIST_DEEP_N3_ALL.csv")
HIST_N3["rep"]=HIST_N3["rep"].replace({"R1":"RT"})

# Train RUT at the exact historical Deep-N3 design.
N3_DIRS={}
for seed in DEEP_N3_SEEDS:
    for B in DEEP_N3_BUDGETS:
        ii=balanced_indices(DEEP_N3_RANK_SEED,B)
        tag=f"N3_seed{seed}_DUAL_RUT_{B}"
        N3_DIRS[(seed,B)]=train_deep(tag,"RUT",B,False,seed,ii,keep_state=True)

# DEV score for new RUT N3; historical R0/RT FINAL values are joined after unlock.
n3_dev_rows=[]
for (seed,B),d in N3_DIRS.items():
    m=load_deep(d,"RUT",False)
    z,bs=score_deep_adaptive(m,DEV,False,"RUT",False)
    s=z["gated"];yc=DEVY[REP_CAL];ye=DEVY[REP_EVAL]
    th=thr_fpr(s[REP_CAL][yc==0],PRIMARY_FPR)
    n3_dev_rows.append({
        "seed":seed,"budget":B,"rep":"RUT","probe":"DEEP_DUAL",
        "target_fpr":PRIMARY_FPR,
        **op(ye,s[REP_EVAL],th),**curves(ye,s[REP_EVAL])
    })
    del m;gc.collect();torch.cuda.empty_cache()
N3_DEV=pd.DataFrame(n3_dev_rows)
N3_DEV.to_csv(RESULTS/"FF2_DEEP_N3_RUT_DEV.csv",index=False)

# Representation champion selection is based ONLY on FF1 DEV Linear Probe.
screen=FF1_AGG[FF1_AGG.probe.eq("LINEAR")].copy()
screen=screen.sort_values(
    ["tpr_mean","P_at_R90_mean","AP_mean"],
    ascending=[False,False,False]
).reset_index(drop=True)
CHAMP=str(screen.iloc[0].rep)

# Engineering 20k/200k DUAL for champion; reuse RUT N3 seed82 if exact condition.
CHAMP_DUAL={}
for B in [20_000,200_000]:
    if CHAMP=="RUT" and (ENGINEERING_SEED,B) in N3_DIRS:
        CHAMP_DUAL[B]=N3_DIRS[(ENGINEERING_SEED,B)]
    else:
        ii=balanced_indices(ENGINEERING_RANK_SEED,B)
        CHAMP_DUAL[B]=train_deep(
            f"ENG_DUAL_{CHAMP}_{B}",CHAMP,B,False,ENGINEERING_SEED,ii
        )

CHAMP_TRI={}
for B in [20_000,200_000]:
    ii=balanced_indices(ENGINEERING_RANK_SEED,B)
    CHAMP_TRI[B]=train_deep(
        f"ENG_TRI_{CHAMP}_{B}",CHAMP,B,True,ENGINEERING_SEED,ii
    )


@torch.no_grad()
def mean_gate_weights(model,df,use_dom,rep,batch=64):
    sums=None;n=0
    url_tokenizer=url_dapt_tok if rep in {"RU","RUT"} else url_tok
    for st in range(0,len(df),batch):
        en=min(st+batch,len(df));rows=df.iloc[st:en]
        u=tok_url_values(rows.url.fillna("").astype(str).tolist(),url_tokenizer)
        t=tok_text_values(rows.text.fillna("").astype(str).tolist())
        g=graph_batch(rows) if use_dom else None
        with amp_ctx():
            _,_,gw=model(u,t,g,"gated")
        x=gw.float().cpu().numpy()
        sums=x.sum(0) if sums is None else sums+x.sum(0);n+=len(x)
    means=sums/max(n,1)
    out={"gate_url_mean":float(means[0]),"gate_text_mean":float(means[1])}
    out["gate_dom_mean"]=float(means[2]) if use_dom else np.nan
    return out

def dev_summary(tag,d,rep,use_dom):
    m=load_deep(d,rep,use_dom)
    a=DEV.iloc[ENG_TUNE].reset_index(drop=True)
    b=DEV.iloc[ENG_META].reset_index(drop=True)
    za,bs=score_deep_adaptive(m,a,use_dom,rep)
    zb,_=score_deep_adaptive(m,b,use_dom,rep)
    yt=DEVY[ENG_TUNE];ym=DEVY[ENG_META]
    th=thr_fpr(za["gated"][yt==0],PRIMARY_FPR)
    gate_means=mean_gate_weights(m,b,use_dom,rep)
    row={
        "tag":tag,"rep":rep,"use_dom":use_dom,
        "threshold":th,"target_fpr":PRIMARY_FPR,
        **op(ym,zb["gated"],th),**curves(ym,zb["gated"]),
        **gate_means,
    }
    del m;gc.collect();torch.cuda.empty_cache()
    return row

dom_rows=[]
for B in [20_000,200_000]:
    dual=dev_summary(f"DUAL_{CHAMP}_{B}",CHAMP_DUAL[B],CHAMP,False)
    tri=dev_summary(f"TRI_{CHAMP}_{B}",CHAMP_TRI[B],CHAMP,True)
    dual["budget"]=B;tri["budget"]=B
    dom_rows.extend([dual,tri])
DOM_DEV=pd.DataFrame(dom_rows)
DOM_DEV.to_csv(TABLES/"TABLE_FF4_DOM_DUAL_TRI_DEV.csv",index=False)

def dom_gate(dualrow,trirow):
    delta_pp=100*(trirow["tpr"]-dualrow["tpr"])
    fpr90_good=trirow["FPR_at_TPR90"]<=.90*dualrow["FPR_at_TPR90"]
    return {
        "use_dom":bool(delta_pp>=.5 or fpr90_good),
        "delta_tpr_pp":float(delta_pp),
        "fpr90_rule_pass":bool(fpr90_good),
        "dual_tpr":float(dualrow["tpr"]),
        "tri_tpr":float(trirow["tpr"]),
        "dual_FPR_at_TPR90":float(dualrow["FPR_at_TPR90"]),
        "tri_FPR_at_TPR90":float(trirow["FPR_at_TPR90"]),
    }

gates={}
for B in [20_000,200_000]:
    dual=DOM_DEV[(DOM_DEV.budget==B)&(~DOM_DEV.use_dom)].iloc[0]
    tri=DOM_DEV[(DOM_DEV.budget==B)&(DOM_DEV.use_dom)].iloc[0]
    gates[str(B)]=dom_gate(dual,tri)

USE_DOM=bool(gates["200000"]["use_dom"])
SELECTED_ARCH="TRI" if USE_DOM else "DUAL"
SELECTED_DIR=CHAMP_TRI[200000] if USE_DOM else CHAMP_DUAL[200000]

# Cascade freeze on DEV.
MSEL=load_deep(SELECTED_DIR,CHAMP,USE_DOM)
tune_df=DEV.iloc[ENG_TUNE].reset_index(drop=True)
meta_df=DEV.iloc[ENG_META].reset_index(drop=True)
ET,BSSEL=score_deep_adaptive(MSEL,tune_df,USE_DOM,CHAMP)
EM,_=score_deep_adaptive(MSEL,meta_df,USE_DOM,CHAMP)
yt=DEVY[ENG_TUNE];ym=DEVY[ENG_META]

full_thr=thr_fpr(ET["gated"][yt==0],PRIMARY_FPR)
url_high=thr_fpr(ET["url"][yt==0],.001)
pos=np.sort(ET["url"][yt==1])
k=max(1,int(math.floor(.01*len(pos))))
url_low=float(np.nextafter(pos[k-1],-np.inf))

full_meta=op(ym,EM["gated"],full_thr)
esc=(EM["url"]>url_low)&(EM["url"]<url_high)
cas=EM["gated"].copy()
cas[EM["url"]>=url_high]=1e9
cas[EM["url"]<=url_low]=-1e9
cas_meta=op(ym,cas,full_thr)
loss_pp=100*(full_meta["tpr"]-cas_meta["tpr"])
USE_CASCADE=bool(
    loss_pp<=CASCADE_MAX_TPR_LOSS_PP and
    esc.mean()<=1-CASCADE_MIN_FULL_REDUCTION and
    cas_meta["fpr"]<=full_meta["fpr"]+.001
)

ARCH_FREEZE={
    "status":"FROZEN_ON_DEV_ONLY",
    "selected_representation":CHAMP,
    "representation_screen":screen.to_dict("records"),
    "dom_gate_20k":gates["20000"],
    "dom_gate_200k":gates["200000"],
    "selected_architecture":SELECTED_ARCH,
    "selected_budget":200000,
    "cascade":{
        "use":USE_CASCADE,
        "url_low":url_low,"url_high":url_high,
        "dev_meta_escalation_rate":float(esc.mean()),
        "dev_meta_full_reduction":float(1-esc.mean()),
        "tpr_loss_pp":float(loss_pp),
        "full_meta":full_meta,"cascade_meta":cas_meta,
    },
    "cal_used":False,"final_used":False,
}
atomic_json(AUDIT/"ARCHITECTURE_FREEZE.json",ARCH_FREEZE)
print(json.dumps(ARCH_FREEZE,indent=2))
del MSEL;gc.collect();torch.cuda.empty_cache()


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 09 — Nach Architecture Freeze: CAL und FINAL freischalten, Integrität/OOD-Masken laden
if not (AUDIT/"ARCHITECTURE_FREEZE.json").exists():
    raise RuntimeError("Architecture Freeze fehlt; CAL/FINAL bleiben gesperrt.")
FINAL_UNLOCKED=True

CAL=read_role("CAL",["sha256","url","text","label","date"])
CAL["sha256"]=CAL.sha256.astype(str).str.lower();CALY=y01(CAL.label)
if len(CAL)!=50000 or int(CALY.sum())!=0:
    raise RuntimeError("CAL muss 50.000 ausschließlich benigne Seiten enthalten.")

FINAL=read_role("FINAL",["sha256","url","text","label","date"])
FINAL["sha256"]=FINAL.sha256.astype(str).str.lower();FINALY=y01(FINAL.label)
if len(FINAL)!=168060 or (int((FINALY==0).sum()),int((FINALY==1).sum()))!=(91260,76800):
    raise RuntimeError("FINAL class counts invalid.")

sets={
    "SSL":set(SSL_META.sha256),"DEV":set(DEV.sha256),
    "CAL":set(CAL.sha256),"FINAL":set(FINAL.sha256)
}
for a,b in itertools.combinations(sets,2):
    if sets[a]&sets[b]:raise RuntimeError(f"SHA overlap: {a}/{b}")

flags=pd.read_parquet(MANIFEST_ROOT/"final_test_exact_ood_flags_SEALED.parquet")
flags["sha256"]=flags.sha256.astype(str).str.lower()
flags=flags.set_index("sha256").loc[FINAL.sha256].reset_index()
dt=pd.to_datetime(FINAL.date,errors="coerce")
cut=dt.dropna().quantile(.75)
late=(dt>=cut).fillna(False).to_numpy()

FINAL_MASKS={
    "OFFICIAL_TEST":np.ones(len(FINAL),dtype=bool),
    "DOMAIN_OOD_EXACT":flags.domain_ood_exact.to_numpy(bool),
    "TEMPLATE_OOD_EXACT":flags.template_ood_exact.to_numpy(bool),
    "DOMAIN_TEMPLATE_OOD_EXACT":flags.domain_template_ood_exact.to_numpy(bool),
    "LATE_TEST_Q4":late,
}
EXPECTED_SCENARIO_COUNTS={
    "OFFICIAL_TEST":168060,
    "DOMAIN_OOD_EXACT":115917,
    "TEMPLATE_OOD_EXACT":161223,
    "DOMAIN_TEMPLATE_OOD_EXACT":110095,
    "LATE_TEST_Q4":46784,
}
got={k:int(v.sum()) for k,v in FINAL_MASKS.items()}
if got!=EXPECTED_SCENARIO_COUNTS:
    raise RuntimeError(f"Scenario count mismatch: {got}")

atomic_json(AUDIT/"FINAL_ACCESS_PROTOCOL.json",{
    "status":"PASS_AFTER_ARCHITECTURE_FREEZE",
    "cal_rows":len(CAL),"cal_phish":int(CALY.sum()),
    "final_rows":len(FINAL),
    "scenario_counts":got,
    "late_q4_cutoff":str(cut),
    "sha_disjointness":"PASS",
})
print({"FINAL_UNLOCK":"PASS","scenario_counts":got})


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 10 — FF2 Final/OOD Label-Effizienz nach CAL-basierter Schwellenwertbestimmung
for stream in STREAMS:
    for role in ["CAL","FINAL"]:
        embed_stream_role(stream,role)

FINAL_CURVE_PARTS=RESULTS/"curve_final_parts";FINAL_CURVE_PARTS.mkdir(exist_ok=True)

def score_saved_probe(run_i,rep,probe,B,role):
    X=take_features(rep,role)
    cdir=PROBE_CKPT/f"run{run_i}"/rep/f"B{B}"
    if probe=="LINEAR":
        z=np.load(cdir/"linear.npz")
        return ((X@z["coef"].T).reshape(-1)+float(z["intercept"].reshape(-1)[0])).astype(np.float32)
    m=MLPProbe(FEATURE_DIM).to(DEVICE)
    m.load_state_dict(torch.load(cdir/"mlp.pt",map_location=DEVICE));m.eval()
    s=score_mlp_features(m,X);del m;torch.cuda.empty_cache()
    return s

for run_i,(mseed,rseed) in enumerate(zip(CURVE_MODEL_SEEDS,CURVE_LABEL_RANK_SEEDS)):
    for B in BUDGETS:
        for rep in ["R0","RU","RT","RUT"]:
            for probe in ["LINEAR","MLP"]:
                part=FINAL_CURVE_PARTS/f"final_run{run_i}_{rep}_{probe}_{B}.csv"
                if part.exists():continue
                cs=score_saved_probe(run_i,rep,probe,B,"CAL")
                fs=score_saved_probe(run_i,rep,probe,B,"FINAL")
                rows=[]
                for f in TARGET_FPRS:
                    th=thr_fpr(cs,f)
                    for sn,mask in FINAL_MASKS.items():
                        rows.append({
                            "run":run_i,"model_seed":mseed,"label_rank_seed":rseed,
                            "budget":B,"rep":rep,"probe":probe,
                            "dataset":"FINAL","scenario":sn,
                            "target_fpr":f,"threshold":th,
                            **op(FINALY[mask],fs[mask],th),
                            **curves(FINALY[mask],fs[mask])
                        })
                pd.DataFrame(rows).to_csv(part,index=False)
                print({"CURVE_FINAL":True,"run":run_i,"B":B,"rep":rep,"probe":probe})

FINAL_CURVE=pd.concat([pd.read_csv(p) for p in sorted(FINAL_CURVE_PARTS.glob("*.csv"))],ignore_index=True)
FINAL_CURVE.to_csv(RESULTS/"FF2_LABEL_CURVE_FINAL_ALL.csv",index=False)

PRIMARY_CURVE=FINAL_CURVE[
    FINAL_CURVE.scenario.eq("OFFICIAL_TEST") &
    np.isclose(FINAL_CURVE.target_fpr,PRIMARY_FPR)
]
CURVE_AGG=PRIMARY_CURVE.groupby(["rep","probe","budget"],as_index=False).agg(
    n=("tpr","size"),
    tpr_mean=("tpr","mean"),tpr_std=("tpr","std"),
    fpr_mean=("fpr","mean"),precision_mean=("precision","mean"),
    AP_mean=("AP","mean"),P_at_R90_mean=("P_at_R90","mean")
)
CURVE_AGG.to_csv(TABLES/"TABLE_FF2_LABEL_EFFICIENCY_FINAL.csv",index=False)

# Paired deltas across the four useful scientific contrasts.
delta_rows=[]
for probe in ["LINEAR","MLP"]:
    for B in BUDGETS:
        g=PRIMARY_CURVE[(PRIMARY_CURVE.probe==probe)&(PRIMARY_CURVE.budget==B)]
        for name,a,b in [
            ("RU-R0","RU","R0"),("RT-R0","RT","R0"),
            ("RUT-R0","RUT","R0"),("RUT-RT","RUT","RT"),("RUT-RU","RUT","RU")
        ]:
            piv=g[g.rep.isin([a,b])].pivot(index="run",columns="rep",values="tpr")
            d=(piv[a]-piv[b]).to_numpy(float)
            m,lo,hi=mean_ci(d)
            delta_rows.append({
                "probe":probe,"budget":B,"contrast":name,
                "mean_delta_tpr_pp":100*m,"ci95_lo_pp":100*lo,"ci95_hi_pp":100*hi,
                "positive_runs":int((d>0).sum()),"negative_runs":int((d<0).sum()),
                "exact_signflip_p":exact_signflip_p(d)
            })
FF2_DELTA=pd.DataFrame(delta_rows)
FF2_DELTA["holm_p"]=np.nan
# Holm separately for each contrast across 7 budgets x 2 probes = 14 tests.
for contrast in FF2_DELTA.contrast.unique():
    mask=FF2_DELTA.contrast.eq(contrast)
    FF2_DELTA.loc[mask,"holm_p"]=holm_adjust(FF2_DELTA.loc[mask,"exact_signflip_p"].to_numpy(float))
FF2_DELTA.to_csv(TABLES/"TABLE_FF2_LABEL_EFFICIENCY_PAIRED_DELTAS.csv",index=False)
display(CURVE_AGG)


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 11 — FF2 Deep-N3: RUT neu auswerten und mit historischen R0/RT-Bedingungen paaren
def load_role_for_deep(key,use_dom=False):
    cols=["sha256","url","text","label"]+(DOM_COLS if use_dom else [])
    q=read_role(key,cols)
    q["sha256"]=q.sha256.astype(str).str.lower()
    return q

def score_dual_cal_final(d,rep,tag):
    base=SCORES/"deep_n3"/tag;base.mkdir(parents=True,exist_ok=True)
    cp=base/"CAL.npy";fp=base/"FINAL.npy"
    if cp.exists() and fp.exists():
        return np.load(cp),np.load(fp)
    m=load_deep(d,rep,False)
    cz,_=score_deep_adaptive(m,CAL,False,rep,False)
    fz,_=score_deep_adaptive(m,FINAL,False,rep,False)
    np.save(cp,cz["gated"]);np.save(fp,fz["gated"])
    del m;gc.collect();torch.cuda.empty_cache()
    return cz["gated"],fz["gated"]

new_rows=[]
for (seed,B),d in N3_DIRS.items():
    cs,fs=score_dual_cal_final(d,"RUT",f"seed{seed}_B{B}")
    for f in TARGET_FPRS:
        th=thr_fpr(cs,f)
        for sn,mask in FINAL_MASKS.items():
            new_rows.append({
                "seed":seed,"rep":"RUT","budget":B,
                "dataset":"FINAL","scenario":sn,
                "target_fpr":f,"threshold":th,
                **op(FINALY[mask],fs[mask],th),
                **curves(FINALY[mask],fs[mask])
            })
N3_NEW=pd.DataFrame(new_rows)
N3_NEW.to_csv(RESULTS/"FF2_DEEP_N3_RUT_FINAL_ALL.csv",index=False)

# Normalize historical schema and preserve exact source rows.
HIST_N3.to_csv(RESULTS/"FF2_DEEP_N3_HISTORICAL_R0_RT.csv",index=False)

# Main reporting slice.
new_primary=N3_NEW[
    N3_NEW.scenario.eq("OFFICIAL_TEST") &
    np.isclose(N3_NEW.target_fpr,PRIMARY_FPR)
].copy()
hist_primary=HIST_N3[
    HIST_N3.scenario.eq("OFFICIAL_TEST") &
    np.isclose(HIST_N3.target_fpr.fillna(-1),PRIMARY_FPR)
].copy()

# Determine historical seed column.
hist_seed_col="seed" if "seed" in hist_primary.columns else "model_seed"
if hist_seed_col not in hist_primary.columns:
    raise RuntimeError("Historical Deep-N3 seed column not found.")

N3_COMBINED=pd.concat([
    hist_primary.rename(columns={hist_seed_col:"seed"}),
    new_primary
],ignore_index=True,sort=False)
N3_COMBINED.to_csv(TABLES/"TABLE_FF2_DEEP_N3_20K_200K.csv",index=False)

# New paired RUT-RT and RUT-R0 on same historical seeds.
rows=[]
for B in DEEP_N3_BUDGETS:
    for baseline in ["R0","RT"]:
        a=new_primary[new_primary.budget.eq(B)][["seed","tpr"]].rename(columns={"tpr":"RUT"})
        b=hist_primary[(hist_primary.budget==B)&(hist_primary.rep==baseline)][[hist_seed_col,"tpr"]].rename(
            columns={hist_seed_col:"seed","tpr":baseline}
        )
        q=a.merge(b,on="seed",validate="one_to_one")
        d=(q.RUT-q[baseline]).to_numpy(float)
        m,lo,hi=mean_ci(d)
        rows.append({
            "budget":B,"contrast":f"RUT-{baseline}","n":len(d),
            "mean_delta_tpr_pp":100*m,"ci95_lo_pp":100*lo,"ci95_hi_pp":100*hi,
            "positive_pairs":int((d>0).sum()),"negative_pairs":int((d<0).sum()),
            "exact_signflip_p":exact_signflip_p(d),
        })
N3_STATS=pd.DataFrame(rows)
N3_STATS.to_csv(TABLES/"TABLE_FF2_DEEP_N3_RUT_PAIRED.csv",index=False)
display(N3_STATS)


# Classical XGBoost reference is independent of URL/HTML Transformer DAPT.
# Preserve its historical v8.4 evidence unchanged and build an updated comparison
# that includes the newly evaluated RUT Deep-DUAL conditions.
XGB_ALL=pd.read_csv(HIST/"HIST_XGB_N3_ALL_BUDGETS_AGG.csv")
XGB_PRIMARY_SEED82=pd.read_csv(HIST/"HIST_XGB_LABEL_CURVE_SEED82_PRIMARY.csv")
XGB_OP_HIST=pd.read_csv(HIST/"HIST_XGB_OPERATIONAL_BENCHMARK.csv")
XGB_SYSTEM_HIST=pd.read_csv(HIST/"HIST_SYSTEM_REFERENCE_XGB_VS_DEEP_N3_PRIMARY.csv")

XGB_ALL.to_csv(TABLES/"TABLE_SYSTEM_REFERENCE_XGB_ALL_BUDGETS_HISTORICAL.csv",index=False)
XGB_PRIMARY_SEED82.to_csv(TABLES/"TABLE_SYSTEM_REFERENCE_XGB_SEED82_PRIMARY_HISTORICAL.csv",index=False)
XGB_OP_HIST.to_csv(TABLES/"TABLE_SYSTEM_REFERENCE_XGB_OPERATIONAL_HISTORICAL.csv",index=False)

# Updated primary system-reference table:
# historical XGB + historical Deep R0/RT + new Deep RUT.
sys_rows=[]
# Take historical XGB rows from canonical historical system-reference table.
sys_rows.extend(XGB_SYSTEM_HIST[XGB_SYSTEM_HIST.system.eq("TFIDF_XGB")].to_dict("records"))

# Preserve historical Deep R0/RT rows at 20k/200k from the canonical system-reference table.
for oldname,newname in [("DEEP_DUAL_R0","DEEP_DUAL_R0"),("DEEP_DUAL_R1","DEEP_DUAL_RT")]:
    q=XGB_SYSTEM_HIST[XGB_SYSTEM_HIST.system.eq(oldname)].copy()
    if len(q):
        q["system"]=newname
        sys_rows.extend(q.to_dict("records"))

# Add new RUT Deep-N3 means at same 20k/200k primary operating point.
for B in DEEP_N3_BUDGETS:
    q=new_primary[new_primary.budget.eq(B)]
    sys_rows.append({
        "budget":B,"system":"DEEP_DUAL_RUT","n":len(q),
        "tpr_mean":float(q.tpr.mean()),"tpr_std":float(q.tpr.std(ddof=1)),
        "fpr_mean":float(q.fpr.mean()),"fpr_std":float(q.fpr.std(ddof=1)),
        "precision_mean":float(q.precision.mean()),
        "AP_mean":float(q.AP.mean()),
        "P_at_R90_mean":float(q.P_at_R90.mean()),
        "low_fpr_comparison_valid":True,
    })
SYSTEM_REF_UPDATED=pd.DataFrame(sys_rows)
SYSTEM_REF_UPDATED.to_csv(TABLES/"TABLE_SYSTEM_REFERENCE_XGB_VS_DEEP_UPDATED.csv",index=False)


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 12 — FF3: N10-Erweiterung mit RU und RUT, exakt dieselben Seed-/Label-Paarungen wie historisch
HIST_N10=pd.read_csv(HIST/"HIST_N10_ALL.csv")
HIST_N10["rep"]=HIST_N10["rep"].replace({"R1":"RT"})
if set(HIST_N10.rep.unique())!={"R0","RT"}:
    raise RuntimeError(f"Historical N10 reps unexpected: {HIST_N10.rep.unique()}")

N10_PARTS=RESULTS/"n10_parts";N10_PARTS.mkdir(exist_ok=True)
N10_SCORE=SCORES/"n10";N10_SCORE.mkdir(exist_ok=True)

def run_n10_condition(rep_i,rep):
    mseed=N10_MODEL_SEEDS[rep_i]
    rseed=N10_LABEL_RANK_SEEDS[rep_i]
    idx=N10_SELECTIONS[rep_i]
    out=N10_PARTS/f"rep{rep_i:02d}_{rep}.csv"
    marker=N10_PARTS/f"rep{rep_i:02d}_{rep}_COMPLETE.json"
    if out.exists() and marker.exists():return

    tag=f"N10_rep{rep_i:02d}_{rep}"
    d=train_deep(tag,rep,N10_BUDGET,False,mseed,idx,keep_state=True)
    sdir=N10_SCORE/f"rep{rep_i:02d}_{rep}";sdir.mkdir(parents=True,exist_ok=True)
    cp=sdir/"CAL.npy";fp=sdir/"FINAL.npy"
    if cp.exists() and fp.exists():
        cs=np.asarray(np.load(cp,mmap_mode="r"))
        fs=np.asarray(np.load(fp,mmap_mode="r"))
    else:
        m=load_deep(d,rep,False)
        cz,_=score_deep_adaptive(m,CAL,False,rep,False)
        fz,_=score_deep_adaptive(m,FINAL,False,rep,False)
        cs=cz["gated"];fs=fz["gated"]
        np.save(cp,cs);np.save(fp,fs)
        del m;gc.collect();torch.cuda.empty_cache()

    th=thr_fpr(cs,PRIMARY_FPR)
    rows=[]
    for sn,mask in FINAL_MASKS.items():
        rows.append({
            "replicate":rep_i,
            "model_seed":mseed,
            "label_rank_seed":rseed,
            "rep":rep,
            "budget":N10_BUDGET,
            "dataset":"FINAL","scenario":sn,
            "target_fpr":PRIMARY_FPR,"threshold":th,
            **op(FINALY[mask],fs[mask],th),
            **curves(FINALY[mask],fs[mask])
        })
    pd.DataFrame(rows).to_csv(out,index=False)
    atomic_json(marker,{"status":"COMPLETE","replicate":rep_i,"rep":rep,"model_seed":mseed,"label_rank_seed":rseed})

    if not KEEP_N10_MODEL_STATES_AFTER_SCORING:
        state=Path(d)/"model_state.pt"
        if state.exists():state.unlink()

for rep_i in range(10):
    for rep in ["RU","RUT"]:
        run_n10_condition(rep_i,rep)
    print({"N10_REPLICATE_COMPLETE":rep_i})

NEW_N10=pd.concat([pd.read_csv(p) for p in sorted(N10_PARTS.glob("rep??_R[U]*.csv"))],ignore_index=True)
# Glob above can be implementation-dependent; rebuild explicitly.
NEW_N10=pd.concat([
    pd.read_csv(N10_PARTS/f"rep{i:02d}_{rep}.csv")
    for i in range(10) for rep in ["RU","RUT"]
],ignore_index=True)
NEW_N10.to_csv(RESULTS/"FF3_N10_NEW_RU_RUT_ALL.csv",index=False)

N10_ALL=pd.concat([HIST_N10,NEW_N10],ignore_index=True,sort=False)
N10_ALL.to_csv(RESULTS/"FF3_N10_R0_RU_RT_RUT_ALL.csv",index=False)

N10_AGG=N10_ALL.groupby(["rep","scenario"],as_index=False).agg(
    n=("tpr","size"),tpr_mean=("tpr","mean"),tpr_std=("tpr","std"),
    fpr_mean=("fpr","mean"),fpr_std=("fpr","std"),
    precision_mean=("precision","mean"),
    AP_mean=("AP","mean"),P_at_R90_mean=("P_at_R90","mean"),
)
N10_AGG.to_csv(TABLES/"TABLE_FF3_N10_4REP_AGG.csv",index=False)

contrast_specs=[
    ("RU-R0","RU","R0"),
    ("RT-R0","RT","R0"),
    ("RUT-R0","RUT","R0"),
    ("RUT-RT","RUT","RT"),
    ("RUT-RU","RUT","RU"),
    ("RU-RT","RU","RT"),
]
stats=[]
for sn in SCENARIOS:
    g=N10_ALL[N10_ALL.scenario.eq(sn)]
    for name,a,b in contrast_specs:
        piv=g[g.rep.isin([a,b])].pivot_table(index="replicate",columns="rep",values="tpr",aggfunc="first")
        if not {a,b}.issubset(piv.columns) or len(piv.dropna())!=10:
            raise RuntimeError(f"N10 incomplete: {sn} {name}")
        d=(piv[a]-piv[b]).to_numpy(float)
        m,lo,hi=mean_ci(d)
        stats.append({
            "scenario":sn,"contrast":name,"n_pairs":len(d),
            "mean_delta_tpr_pp":100*m,"ci95_lo_pp":100*lo,"ci95_hi_pp":100*hi,
            "positive_pairs":int((d>0).sum()),"negative_pairs":int((d<0).sum()),
            "exact_signflip_p":exact_signflip_p(d),
        })
N10_STATS=pd.DataFrame(stats)
N10_STATS["holm_ood_p"]=np.nan
# Holm across exactly four OOD/Late scenarios separately per contrast.
for contrast in N10_STATS.contrast.unique():
    mask=N10_STATS.contrast.eq(contrast)&N10_STATS.scenario.isin(OOD_FAMILY)
    N10_STATS.loc[mask,"holm_ood_p"]=holm_adjust(
        N10_STATS.loc[mask,"exact_signflip_p"].to_numpy(float)
    )
N10_STATS["primary_extension_test"]=(
    N10_STATS.contrast.eq("RUT-RT")&N10_STATS.scenario.eq("OFFICIAL_TEST")
)
N10_STATS["primary_extension_ood_family"]=(
    N10_STATS.contrast.eq("RUT-RT")&N10_STATS.scenario.isin(OOD_FAMILY)
)
N10_STATS.to_csv(TABLES/"TABLE_FF3_N10_PAIRED_CONTRASTS.csv",index=False)
N10_STATS[N10_STATS.primary_extension_test].to_csv(
    TABLES/"TABLE_FF3_PRIMARY_RUT_MINUS_RT_OFFICIAL.csv",index=False
)
N10_STATS[N10_STATS.primary_extension_ood_family].to_csv(
    TABLES/"TABLE_FF3_OOD_HOLM_RUT_MINUS_RT.csv",index=False
)
display(N10_STATS[N10_STATS.contrast.eq("RUT-RT")])


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 13 — FF4: neue DUAL/TRI/Fusion/Cascade-FINAL-Auswertung mit dem DEV-gefrorenen Champion
def score_deep_cache(tag,d,rep,use_dom,key):
    base=SCORES/"ff4"/tag;base.mkdir(parents=True,exist_ok=True)
    modes=["gated","equal","url","text"]+(["dom"] if use_dom else [])
    paths={m:base/f"{key}_{m}.npy" for m in modes}
    done=base/f"{key}.complete.json"
    if done.exists() and all(p.exists() for p in paths.values()):return paths

    m=load_deep(d,rep,use_dom)
    df=CAL if key=="CAL" else FINAL
    # For TRI, materialize DOM in chunks to avoid a giant nested Arrow object in memory.
    arrays={k:np.lib.format.open_memmap(p,mode="w+",dtype=np.float32,shape=(len(df),)) for k,p in paths.items()}
    chunk=10000
    for st in range(0,len(df),chunk):
        en=min(st+chunk,len(df))
        x=df.iloc[st:en].copy().reset_index(drop=True)
        if use_dom:
            ddom=read_role_slice(key,st,en,["sha256"]+DOM_COLS)
            ddom["sha256"]=ddom.sha256.astype(str).str.lower()
            if not np.array_equal(x.sha256.to_numpy(),ddom.sha256.to_numpy()):
                raise RuntimeError(f"{key} DOM SHA alignment failed {st}:{en}")
            for c in DOM_COLS:x[c]=ddom[c].tolist()
        z,_=score_deep_adaptive(m,x,use_dom,rep,True)
        for mode in modes:
            arrays[mode][st:en]=z[mode].astype(np.float32);arrays[mode].flush()
        print({"FF4_SCORE":tag,"role":key,"rows":en,"total":len(df)})
    for a in arrays.values():a.flush()
    del arrays,m;gc.collect();torch.cuda.empty_cache()
    atomic_json(done,{"status":"COMPLETE","rows":len(df),"raw_fp32_logits":True})
    return paths

# Score DUAL/TRI at 20k and 200k for champion.
FF4_SYSTEMS={}
for B in [20_000,200_000]:
    FF4_SYSTEMS[f"DUAL_{CHAMP}_{B}"]=(CHAMP_DUAL[B],False)
    FF4_SYSTEMS[f"TRI_{CHAMP}_{B}"]=(CHAMP_TRI[B],True)

CACHE={}
for tag,(d,use_dom) in FF4_SYSTEMS.items():
    CACHE[tag]=(
        score_deep_cache(tag,d,CHAMP,use_dom,"CAL"),
        score_deep_cache(tag,d,CHAMP,use_dom,"FINAL")
    )

ff4_rows=[]
for tag,(d,use_dom) in FF4_SYSTEMS.items():
    cp,fp=CACHE[tag]
    modes=["url","text","gated","equal"]+(["dom"] if use_dom else [])
    B=int(json.loads((Path(d)/"COMPLETE.json").read_text())["budget"])
    for mode in modes:
        cs=np.asarray(np.load(cp[mode],mmap_mode="r"))
        fs=np.asarray(np.load(fp[mode],mmap_mode="r"))
        for f in TARGET_FPRS:
            th=thr_fpr(cs,f)
            for sn,mask in FINAL_MASKS.items():
                ff4_rows.append({
                    "system":tag,"rep":CHAMP,"budget":B,"use_dom":use_dom,
                    "mode":mode,"scenario":sn,"target_fpr":f,"threshold":th,
                    **op(FINALY[mask],fs[mask],th),
                    **curves(FINALY[mask],fs[mask])
                })

selected_tag=f"{SELECTED_ARCH}_{CHAMP}_200000"
cp,fp=CACHE[selected_tag]
full_cal=np.asarray(np.load(cp["gated"],mmap_mode="r"))
url_cal=np.asarray(np.load(cp["url"],mmap_mode="r"))
full_fin=np.asarray(np.load(fp["gated"],mmap_mode="r"))
url_fin=np.asarray(np.load(fp["url"],mmap_mode="r"))

if USE_CASCADE:
    ccal=full_cal.copy();cfin=full_fin.copy()
    ccal[url_cal>=url_high]=1e9;ccal[url_cal<=url_low]=-1e9
    cfin[url_fin>=url_high]=1e9;cfin[url_fin<=url_low]=-1e9
    for f in TARGET_FPRS:
        th=thr_fpr(ccal,f)
        for sn,mask in FINAL_MASKS.items():
            ff4_rows.append({
                "system":selected_tag,"rep":CHAMP,"budget":200000,
                "use_dom":USE_DOM,"mode":"cascade","scenario":sn,
                "target_fpr":f,"threshold":th,
                **op(FINALY[mask],cfin[mask],th),
                **curves(FINALY[mask],cfin[mask])
            })
    FINAL_ESC=float(((url_fin>url_low)&(url_fin<url_high)).mean())
else:
    ccal=full_cal;cfin=full_fin;FINAL_ESC=1.0

FF4_FINAL=pd.DataFrame(ff4_rows)
FF4_FINAL.to_csv(TABLES/"TABLE_FF4_DOM_FUSION_CASCADE_FINAL.csv",index=False)

# Compact primary 0.5% OFFICIAL table.
FF4_PRIMARY=FF4_FINAL[
    FF4_FINAL.scenario.eq("OFFICIAL_TEST")&
    np.isclose(FF4_FINAL.target_fpr,PRIMARY_FPR)
].copy()
FF4_PRIMARY.to_csv(TABLES/"TABLE_FF4_PRIMARY_OFFICIAL_0p5FPR.csv",index=False)

print({"FINAL_CASCADE_ESCALATION":FINAL_ESC,"selected_system":selected_tag})
display(FF4_PRIMARY)


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 14 — Operational Benchmark: gleiche Ergebnisbreite wie historisch
try:
    import psutil
except Exception:
    psutil=None

DEV_OP=DEV[["sha256","url","text","label"]+DOM_COLS].copy()
bench_order=np.argsort(
    np.array([int(hashlib.sha256(x.encode()).hexdigest()[:16],16) for x in DEV_OP.sha256],dtype=np.uint64)
)
DEV_B1=DEV_OP.iloc[bench_order[:min(BENCH_B1_N,len(DEV_OP))]].reset_index(drop=True)
DEV_TPUT=DEV_OP.iloc[bench_order[:min(BENCH_TPUT_N,len(DEV_OP))]].reset_index(drop=True)

@torch.no_grad()
def forward_url_branch(m,rows,rep):
    tokenizer=url_dapt_tok if rep in {"RU","RUT"} else url_tok
    u=tok_url_values(rows.url.fillna("").astype(str).tolist(),tokenizer)
    with amp_ctx():
        zu=F.normalize(m.ua(pooled(m.u(**u),u["attention_mask"])),dim=-1)
        s=m.uaux(zu).squeeze(-1)
    return s.float()

@torch.no_grad()
def forward_full(m,rows,use_dom,rep):
    tokenizer=url_dapt_tok if rep in {"RU","RUT"} else url_tok
    u=tok_url_values(rows.url.fillna("").astype(str).tolist(),tokenizer)
    t=tok_text_values(rows.text.fillna("").astype(str).tolist())
    g=graph_batch(rows) if use_dom else None
    with amp_ctx():s,_,_=m(u,t,g)
    return s.float()

@torch.no_grad()
def forward_cascade(m,rows,use_dom,rep):
    tokenizer=url_dapt_tok if rep in {"RU","RUT"} else url_tok
    u=tok_url_values(rows.url.fillna("").astype(str).tolist(),tokenizer)
    with amp_ctx():
        zu=F.normalize(m.ua(pooled(m.u(**u),u["attention_mask"])),dim=-1)
        us=m.uaux(zu).squeeze(-1)
    out=torch.empty_like(us,dtype=torch.float32)
    hi=us>=url_high;lo=us<=url_low;esc=~(hi|lo)
    out[hi]=1e9;out[lo]=-1e9
    if esc.any():
        ids=torch.where(esc)[0].cpu().numpy()
        sub=rows.iloc[ids].reset_index(drop=True)
        zue=zu[esc]
        t=tok_text_values(sub.text.fillna("").astype(str).tolist())
        if use_dom:
            g=graph_batch(sub)
            with amp_ctx():
                zt=F.normalize(m.ta(pooled(m.t(**t),t["attention_mask"])),dim=-1)
                zd=F.normalize(m.da(m.dom(g)),dim=-1)
                zs=[zue,zt,zd];st=torch.stack(zs,1)
                gw=torch.softmax(m.gate(st).squeeze(-1),1)
                weighted=(st*gw[:,:,None]).sum(1)
                main=m.head(torch.cat(zs+[weighted],1)).squeeze(-1)
        else:
            with amp_ctx():
                zt=F.normalize(m.ta(pooled(m.t(**t),t["attention_mask"])),dim=-1)
                zs=[zue,zt];st=torch.stack(zs,1)
                gw=torch.softmax(m.gate(st).squeeze(-1),1)
                weighted=(st*gw[:,:,None]).sum(1)
                main=m.head(torch.cat(zs+[weighted],1)).squeeze(-1)
        out[esc]=main.float()
    return out,float(esc.float().mean().item())

class GPUPoller:
    def __init__(self,interval=.10):
        self.interval=interval;self.stop_evt=threading.Event();self.util=[];self.mem=[]
        self.thread=None
    def _run(self):
        while not self.stop_evt.is_set():
            try:
                r=subprocess.run(
                    ["nvidia-smi","--query-gpu=utilization.gpu,memory.used",
                     "--format=csv,noheader,nounits"],
                    capture_output=True,text=True,timeout=2
                )
                if r.returncode==0 and r.stdout.strip():
                    a=r.stdout.strip().splitlines()[0].split(",")
                    self.util.append(float(a[0].strip()));self.mem.append(float(a[1].strip()))
            except Exception:
                pass
            self.stop_evt.wait(self.interval)
    def start(self):
        self.thread=threading.Thread(target=self._run,daemon=True);self.thread.start()
    def stop(self):
        self.stop_evt.set()
        if self.thread:self.thread.join(timeout=2)
        return (
            float(np.mean(self.util)) if self.util else np.nan,
            float(np.mean(self.mem)) if self.mem else np.nan
        )

def bench_system(name,m,mode,rows_b1,rows_tput,batch,use_dom,rep,model_state_bytes):
    for _ in range(3):
        x=rows_b1.iloc[:min(8,len(rows_b1))]
        if mode=="url":forward_url_branch(m,x,rep)
        elif mode=="cascade":forward_cascade(m,x,use_dom,rep)
        else:forward_full(m,x,use_dom,rep)
    torch.cuda.synchronize()

    # B1 latency + peak VRAM.
    torch.cuda.reset_peak_memory_stats()
    lat=[]
    for i in range(len(rows_b1)):
        x=rows_b1.iloc[i:i+1]
        t0=time.perf_counter()
        if mode=="url":forward_url_branch(m,x,rep)
        elif mode=="cascade":forward_cascade(m,x,use_dom,rep)
        else:forward_full(m,x,use_dom,rep)
        torch.cuda.synchronize()
        lat.append((time.perf_counter()-t0)*1000)
    peak_b1=torch.cuda.max_memory_allocated()/1024**2

    # Throughput + peak VRAM + GPU telemetry.
    torch.cuda.reset_peak_memory_stats()
    poll=GPUPoller(GPU_POLL_S);poll.start()
    times=[];esc_sum=0.
    for _ in range(BENCH_REPEATS):
        t0=time.perf_counter()
        for st in range(0,len(rows_tput),batch):
            x=rows_tput.iloc[st:st+batch]
            if mode=="url":forward_url_branch(m,x,rep)
            elif mode=="cascade":
                _,e=forward_cascade(m,x,use_dom,rep);esc_sum+=e*len(x)
            else:forward_full(m,x,use_dom,rep)
        torch.cuda.synchronize();times.append(time.perf_counter()-t0)
    gpu_util,gpu_mem=poll.stop()
    peak_tput=torch.cuda.max_memory_allocated()/1024**2

    return {
        "system":name,"mode":mode,"batch_throughput":batch,
        "n_b1":len(rows_b1),"n_throughput":len(rows_tput),"repeats":BENCH_REPEATS,
        "latency_b1_median_ms":float(np.median(lat)),
        "latency_b1_p95_ms":float(np.percentile(lat,95)),
        "throughput_pages_s_mean":float(np.mean([len(rows_tput)/t for t in times])),
        "throughput_pages_s_std":float(np.std([len(rows_tput)/t for t in times],ddof=1)),
        "peak_vram_b1_allocated_mb":float(peak_b1),
        "peak_vram_throughput_allocated_mb":float(peak_tput),
        "gpu_util_mean_pct":gpu_util,
        "nvidia_memory_used_mean_mb":gpu_mem,
        "cascade_escalation_rate_sample":float(esc_sum/(len(rows_tput)*BENCH_REPEATS)) if mode=="cascade" else np.nan,
        "model_state_bytes":int(model_state_bytes),
    }

selected_state=Path(SELECTED_DIR)/"model_state.pt"
selected_state_bytes=selected_state.stat().st_size
dual_state=Path(CHAMP_DUAL[200000])/"model_state.pt"
dual_state_bytes=dual_state.stat().st_size

ops=[]
m=load_deep(SELECTED_DIR,CHAMP,USE_DOM)
ops.append(bench_system(f"URL_ONLY_FROM_{SELECTED_ARCH}_{CHAMP}",m,"url",DEV_B1,DEV_TPUT,BENCH_BATCH,USE_DOM,CHAMP,selected_state_bytes))
ops.append(bench_system(f"{SELECTED_ARCH}_{CHAMP}_200K_FULL",m,"full",DEV_B1,DEV_TPUT,BENCH_BATCH,USE_DOM,CHAMP,selected_state_bytes))
if USE_CASCADE:
    ops.append(bench_system(f"{SELECTED_ARCH}_{CHAMP}_200K_CASCADE",m,"cascade",DEV_B1,DEV_TPUT,BENCH_BATCH,USE_DOM,CHAMP,selected_state_bytes))
del m;gc.collect();torch.cuda.empty_cache()

m=load_deep(CHAMP_DUAL[200000],CHAMP,False)
ops.append(bench_system(f"DUAL_{CHAMP}_200K_FULL",m,"full",DEV_B1,DEV_TPUT,BENCH_BATCH,False,CHAMP,dual_state_bytes))
del m;gc.collect();torch.cuda.empty_cache()

OPS=pd.DataFrame(ops)
full_name=f"{SELECTED_ARCH}_{CHAMP}_200K_FULL"
base=float(OPS.loc[OPS.system.eq(full_name),"throughput_pages_s_mean"].iloc[0])
OPS["vs_selected_full_throughput_speedup"]=OPS.throughput_pages_s_mean/base
OPS["hardware"]=torch.cuda.get_device_name(0)
OPS["measurement_scope"]="in-memory URL/text/stored-DOM -> model forward; excludes disk/network/browser/raw HTML parsing"
OPS.to_csv(TABLES/"TABLE_OPERATIONAL_BENCHMARK.csv",index=False)

# Historical independent XGB operational reference carried forward unchanged.
pd.read_csv(HIST/"HIST_XGB_OPERATIONAL_BENCHMARK.csv").to_csv(
    TABLES/"TABLE_OPERATIONAL_XGB_HISTORICAL.csv",index=False
)

# Illustration only: 1M pages, not a production-load claim.
if USE_CASCADE:
    full_tput=float(OPS.loc[OPS.system.eq(full_name),"throughput_pages_s_mean"].iloc[0])
    cas_name=f"{SELECTED_ARCH}_{CHAMP}_200K_CASCADE"
    cas_tput=float(OPS.loc[OPS.system.eq(cas_name),"throughput_pages_s_mean"].iloc[0])
    million=pd.DataFrame([{
        "pages":1_000_000,
        "full_minutes":1_000_000/full_tput/60,
        "cascade_minutes":1_000_000/cas_tput/60,
        "minutes_saved":1_000_000/full_tput/60-1_000_000/cas_tput/60,
        "final_escalation_rate":FINAL_ESC,
        "full_path_executions_avoided":int(round(1_000_000*(1-FINAL_ESC))),
        "scope":"illustrative extrapolation from measured in-memory throughput; not production load",
    }])
    million.to_csv(TABLES/"TABLE_OPERATIONAL_1M_ILLUSTRATION.csv",index=False)

display(OPS)


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 15 — Finale Unsicherheitsintervalle für ausgewähltes Full-System und Cascade
rng=np.random.default_rng(BOOTSTRAP_SEED)
y=FINALY
pos=np.where(y==1)[0];neg=np.where(y==0)[0]

systems={"FULL":full_fin}
if USE_CASCADE:systems["CASCADE"]=cfin

boot_rows=[]
for name,scores in systems.items():
    ap=[];p90=[]
    for b in range(BOOTSTRAP_REPS):
        ip=rng.choice(pos,size=len(pos),replace=True)
        inn=rng.choice(neg,size=len(neg),replace=True)
        ids=np.concatenate([ip,inn])
        yy=y[ids];ss=scores[ids]
        ap.append(average_precision_score(yy,ss))
        p90.append(precision_at_recall(yy,ss,.90))
    boot_rows.append({
        "system":name,"reps":BOOTSTRAP_REPS,
        "AP":float(average_precision_score(y,scores)),
        "AP_ci95_lo":float(np.percentile(ap,2.5)),
        "AP_ci95_hi":float(np.percentile(ap,97.5)),
        "P_at_R90":float(precision_at_recall(y,scores,.90)),
        "P_at_R90_ci95_lo":float(np.percentile(p90,2.5)),
        "P_at_R90_ci95_hi":float(np.percentile(p90,97.5)),
    })
BOOT=pd.DataFrame(boot_rows)
BOOT.to_csv(TABLES/"TABLE_FINAL_BOOTSTRAP_AP_P90.csv",index=False)
display(BOOT)


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 16 — Finale Forschungszusammenfassung: FF1–FF4, Systementscheidung und Geltungsbereich
def rowdict(df, **conds):
    q=df.copy()
    for k,v in conds.items():
        if isinstance(v,float):
            q=q[np.isclose(q[k],v)]
        else:q=q[q[k].eq(v)]
    return q.iloc[0].to_dict() if len(q) else None


# Explicit completeness audit against the historical research-result dimensions.
coverage_rows=[
    ("FF1_modality_2x2","R0/RU/RT/RUT; N5; Linear+MLP",True),
    ("FF1_contrastive","historical R2C/R3C retained unchanged",True),
    ("FF2_label_budgets","2k,5k,10k,20k,50k,100k,200k",BUDGETS==[2000,5000,10000,20000,50000,100000,200000]),
    ("FF2_representations","R0,RU,RT,RUT",True),
    ("FF2_probes","Linear, MLP",True),
    ("FF2_B95","DEV B95 for all four reps/probes",(TABLES/"TABLE_FF2_B95_DEV.csv").exists()),
    ("FF2_marginal_label_utility","all four reps/probes",(TABLES/"TABLE_FF2_MARGINAL_LABEL_UTILITY_DEV.csv").exists()),
    ("FF2_deep_N3","20k and 200k; RUT + historical R0/RT",(TABLES/"TABLE_FF2_DEEP_N3_20K_200K.csv").exists()),
    ("XGB_all_budgets","historical independent TF-IDF/XGB system reference",(TABLES/"TABLE_SYSTEM_REFERENCE_XGB_ALL_BUDGETS_HISTORICAL.csv").exists()),
    ("XGB_vs_Deep_updated","includes new Deep RUT at 20k/200k",(TABLES/"TABLE_SYSTEM_REFERENCE_XGB_VS_DEEP_UPDATED.csv").exists()),
    ("FF3_scenarios","OFFICIAL, Domain, Template, Domain+Template, Late",set(SCENARIOS)==set(FINAL_MASKS)),
    ("FF3_N10","historical R0/RT + new RU/RUT; same 10 pairings",(TABLES/"TABLE_FF3_N10_4REP_AGG.csv").exists()),
    ("FPR_grid","0.01%,0.05%,0.1%,0.25%,0.5%,1%,2%",TARGET_FPRS==[.0001,.0005,.001,.0025,.005,.01,.02]),
    ("FF4_DUAL_TRI_budgets","20k and 200k",True),
    ("FF4_branch_modes","URL,Text,Equal,Gated plus DOM for TRI",True),
    ("FF4_gate_means","URL/Text/DOM mean fusion weights on DEV",all(x in DOM_DEV.columns for x in ["gate_url_mean","gate_text_mean","gate_dom_mean"])),
    ("FF4_DOM_gate","same historical rule reapplied on DEV",True),
    ("FF4_cascade_definition","single historical URL-first policy re-frozen on DEV",True),
    ("FF4_cascade_final","selected 200k architecture; all FPRs and 5 scenarios",True),
    ("Operational_full_metrics","latency,p95,throughput,VRAM,GPU telemetry,model bytes,escalation",(TABLES/"TABLE_OPERATIONAL_BENCHMARK.csv").exists()),
    ("Operational_XGB_reference","historical unchanged",(TABLES/"TABLE_OPERATIONAL_XGB_HISTORICAL.csv").exists()),
    ("Operational_1M_illustration","if cascade selected",((TABLES/"TABLE_OPERATIONAL_1M_ILLUSTRATION.csv").exists() if USE_CASCADE else True)),
    ("Bootstrap","1000 reps AP/P@R90",(TABLES/"TABLE_FINAL_BOOTSTRAP_AP_P90.csv").exists()),
]
COVERAGE=pd.DataFrame(coverage_rows,columns=["block","coverage","pass"])
COVERAGE.to_csv(TABLES/"TABLE_COMPLETE_COVERAGE_AUDIT.csv",index=False)
atomic_json(AUDIT/"COMPLETE_COVERAGE_AUDIT.json",{
    "status":"PASS" if bool(COVERAGE["pass"].all()) else "FAIL",
    "all_pass":bool(COVERAGE["pass"].all()),
    "rows":COVERAGE.to_dict("records"),
})
if not bool(COVERAGE["pass"].all()):
    raise RuntimeError("Complete coverage audit failed.")

summary={
    "status":"COMPLETE",
    "version":VERSION,
    "scientific_status":"post-hoc modality-specific integration; fixed before this complete execution",
    "url_dapt":DAPT_INFO,
    "selected_representation_dev":CHAMP,
    "selected_architecture_dev":SELECTED_ARCH,
    "use_dom":USE_DOM,
    "use_cascade":USE_CASCADE,
    "cascade_dev":ARCH_FREEZE["cascade"],
    "ff1":{
        "modality_table":str(TABLES/"TABLE_FF1_MODALITY_DAPT_DEV_N5.csv"),
        "paired_stats":str(TABLES/"TABLE_FF1_MODALITY_PAIRED_STATS.csv"),
        "contrastive_historical_table":str(TABLES/"TABLE_FF1_CONTRASTIVE_HISTORICAL_PRIMARY.csv"),
    },
    "ff2":{
        "label_efficiency":str(TABLES/"TABLE_FF2_LABEL_EFFICIENCY_FINAL.csv"),
        "b95_dev":str(TABLES/"TABLE_FF2_B95_DEV.csv"),
        "paired_label_deltas":str(TABLES/"TABLE_FF2_LABEL_EFFICIENCY_PAIRED_DELTAS.csv"),
        "deep_n3":str(TABLES/"TABLE_FF2_DEEP_N3_20K_200K.csv"),
        "deep_n3_rut_paired":str(TABLES/"TABLE_FF2_DEEP_N3_RUT_PAIRED.csv"),
    },
    "ff3":{
        "n10_4rep_agg":str(TABLES/"TABLE_FF3_N10_4REP_AGG.csv"),
        "n10_all_contrasts":str(TABLES/"TABLE_FF3_N10_PAIRED_CONTRASTS.csv"),
        "primary_extension":"RUT-RT OFFICIAL_TEST",
        "primary_table":str(TABLES/"TABLE_FF3_PRIMARY_RUT_MINUS_RT_OFFICIAL.csv"),
        "ood_holm_table":str(TABLES/"TABLE_FF3_OOD_HOLM_RUT_MINUS_RT.csv"),
        "historical_RT_R0_retained":True,
    },
    "ff4":{
        "dom_dev":str(TABLES/"TABLE_FF4_DOM_DUAL_TRI_DEV.csv"),
        "final_all":str(TABLES/"TABLE_FF4_DOM_FUSION_CASCADE_FINAL.csv"),
        "primary":str(TABLES/"TABLE_FF4_PRIMARY_OFFICIAL_0p5FPR.csv"),
        "operational":str(TABLES/"TABLE_OPERATIONAL_BENCHMARK.csv"),
        "bootstrap":str(TABLES/"TABLE_FINAL_BOOTSTRAP_AP_P90.csv"),
        "xgb_reference":str(TABLES/"TABLE_SYSTEM_REFERENCE_XGB_VS_DEEP_UPDATED.csv"),
        "marginal_label_utility":str(TABLES/"TABLE_FF2_MARGINAL_LABEL_UTILITY_DEV.csv"),
        "coverage_audit":str(TABLES/"TABLE_COMPLETE_COVERAGE_AUDIT.csv"),
    },
    "access_integrity":{
        "cal_final_unlocked_only_after_architecture_freeze":True,
        "ssl_physical_no_labels":True,
        "historical_final_known_before_extension":True,
    },
    "interpretation_rules":[
        "R0/RU/RT/RUT quantify modality-specific DAPT; contrastive historical results remain a separate SSL-objective comparison.",
        "The primary new FF3 extension test is RUT-RT, not a retroactive replacement of the historical RT-R0 N10 result.",
        "N10 varies downstream label samples/training seeds with fixed SSL checkpoints; it does not estimate SSL-pretraining randomness.",
        "DOM and cascade remain engineering components and are not mixed into the scientific representation ablation.",
        "Operational timings cover the in-memory model path only."
    ],
}
atomic_json(RESULTS/"FINAL_RESEARCH_SUMMARY.json",summary)

# Final integrity assertions.
required=[
    AUDIT/"INTEGRATED_PROTOCOL.json",
    AUDIT/"ARCHITECTURE_FREEZE.json",
    AUDIT/"FINAL_ACCESS_PROTOCOL.json",
    TABLES/"TABLE_FF1_MODALITY_DAPT_DEV_N5.csv",
    TABLES/"TABLE_FF2_LABEL_EFFICIENCY_FINAL.csv",
    TABLES/"TABLE_FF3_N10_PAIRED_CONTRASTS.csv",
    TABLES/"TABLE_FF4_PRIMARY_OFFICIAL_0p5FPR.csv",
    TABLES/"TABLE_OPERATIONAL_BENCHMARK.csv",
    TABLES/"TABLE_FF2_MARGINAL_LABEL_UTILITY_DEV.csv",
    TABLES/"TABLE_SYSTEM_REFERENCE_XGB_ALL_BUDGETS_HISTORICAL.csv",
    TABLES/"TABLE_SYSTEM_REFERENCE_XGB_VS_DEEP_UPDATED.csv",
    TABLES/"TABLE_COMPLETE_COVERAGE_AUDIT.csv",
]
missing=[str(p) for p in required if not p.exists()]
if missing:raise RuntimeError(f"Final required artifacts missing: {missing}")

COMPLETE={
    "status":"COMPLETE",
    "version":VERSION,
    "completed_utc":pd.Timestamp.utcnow().isoformat(),
    "selected_representation":CHAMP,
    "selected_architecture":SELECTED_ARCH,
    "use_cascade":USE_CASCADE,
    "all_planned_blocks_reported":True,
    "required_artifacts_present":True,
}
atomic_json(ROOT/"FINAL_INTEGRATED_COMPLETE.json",COMPLETE)
print(json.dumps(COMPLETE,indent=2))


In [ ]:
# KI-Unterstuetzung: siehe Erklaerung der Arbeit und Herkunftszelle.

# 17 — Kompaktes finales Ergebnispaket
zip_path=Path("/kaggle/working/FINAL_URL_SSL_INTEGRATED_RESULTS_v2_COMPLETE.zip") if Path("/kaggle/working").exists() else ROOT.parent/"FINAL_URL_SSL_INTEGRATED_RESULTS_v2_COMPLETE.zip"

with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    include_roots=[AUDIT,RESULTS,TABLES]
    for base in include_roots:
        for p in base.rglob("*"):
            if p.is_file():
                z.write(p,arcname=str(p.relative_to(ROOT)))
    for p in [DAPT_DONE,ROOT/"FINAL_INTEGRATED_COMPLETE.json"]:
        if p.exists():z.write(p,arcname=str(p.relative_to(ROOT)))

sha=hashlib.sha256(zip_path.read_bytes()).hexdigest()
atomic_json(ROOT/"FINAL_RESULTS_ZIP_SHA256.json",{"file":str(zip_path),"sha256":sha})
print({
    "FINAL_RESULTS_ZIP":str(zip_path),
    "sha256":sha,
    "size_MiB":round(zip_path.stat().st_size/1024**2,2),
    "checkpoint_note":"URL-DAPT and final model checkpoints stay in the working output; results ZIP contains compact evidence/tables."
})
